# 概率叠加可视化：完整的预测质量与不确定性分析

**目标**: 全面可视化3D脑MRI分割的预测质量、不确定性、校准性能、混淆模式和边界效应

**可视化内容**:
- **A图**: 概率叠加（Top-1类别 + 置信度透明度）
- **B图**: 不确定性热图（熵图 + 边际差）
- **C图**: 可靠性图（温度缩放前后对比，ECE改进）
- **D图**: 风险-覆盖曲线（选择性预测，AURC）
- **E图**: 软混淆邻接热图（空间邻接ROI对混淆分析）
- **F图**: 部分体素指数（PV Index）边界条带评估

**透明度策略**: `alpha = clamp((p1 - p2) / τ, 0, 1)`，其中 τ ≈ 0.4

---

## 1. 路径配置与标签加载

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.patches import Rectangle, Patch
import h5py
import pandas as pd
from collections import Counter
from scipy.stats import entropy

# ============================================================================
# 路径配置 - 修改此处切换被试
# ============================================================================

SUBJECT_ID = "ODP_01_qhlazec"
SPLIT_TYPE = "test"

BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")

if SPLIT_TYPE == "test":
    fold_candidates = list(BASE_RESULT_DIR.glob(f"*_test_{SUBJECT_ID}"))
    FOLD_DIR = fold_candidates[0] if len(fold_candidates) > 0 else BASE_RESULT_DIR / f"fold_01_test_{SUBJECT_ID}"
else:
    FOLD_DIR = BASE_RESULT_DIR / "fold_01_test_ODP_01_qhlazec"

DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")
DOWNSAMPLED_FILE = DATA_ROOT / f"{SUBJECT_ID}_downsampled.npz"
PRED_SOFTMAX_FILE = FOLD_DIR / "pred_3d" / f"{SPLIT_TYPE}_{SUBJECT_ID}_pred_softmax_3d.npz"
GT_3D_FILE = DATA_ROOT / "3d" / f"{SUBJECT_ID}_3d.npz"
LABEL_EXCEL = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")

OUTPUT_DIR = FOLD_DIR / "figs" / "confidence_overlay"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHANNEL_MPRAGE = 341
CHANNEL_QSM = 350

# ============================================================================
# 可视化参数
# ============================================================================

TAU = 0.4
N_CLASSES = 102
SLICE_AXIS = 'axial'
AUTO_SELECT_SLICES = True
MANUAL_SLICES = [30, 50, 70]
CONTOUR_LEVELS = [0.3]
CONTOUR_LINEWIDTH = 1.5

ZOOM_REGIONS = {
    "cortex_white": {"z": None, "x": None, "y": None, "hw": 20},
    "basal_ganglia": {"z": None, "x": None, "y": None, "hw": 20},
    "brainstem": {"z": None, "x": None, "y": None, "hw": 20}
}

print("✓ 路径配置完成")
print(f"  被试: {SUBJECT_ID}, 类型: {SPLIT_TYPE}")
print(f"  Fold: {FOLD_DIR.name}")
print(f"  输出: {OUTPUT_DIR}")

# ============================================================================
# 加载FreeSurfer标签映射
# ============================================================================

print("\n加载FreeSurfer标签映射...")
label_df = pd.read_excel(LABEL_EXCEL)
valid_labels = label_df[label_df['one_hot_loc_alex_label'] != '[]'].copy()
valid_labels['label_idx'] = valid_labels['one_hot_loc_alex_label'].astype(int)

LABEL_NAMES = {0: 'Background'}
LABEL_COLORS_RGB = {0: (0, 0, 0)}

for idx, row in valid_labels.iterrows():
    label_idx = int(row['label_idx'])
    if 1 <= label_idx <= N_CLASSES and label_idx not in LABEL_NAMES:
        LABEL_NAMES[label_idx] = row['tissue_name'].strip("'")
        LABEL_COLORS_RGB[label_idx] = (row['R']/255, row['G']/255, row['B']/255)

colors_list = [LABEL_COLORS_RGB.get(i, (0, 0, 0)) for i in range(N_CLASSES)]
cmap_freesurfer = mcolors.ListedColormap(colors_list)

print(f"✓ 加载了 {len(LABEL_NAMES)} 个标签")

## 2. 加载数据并计算置信度与不确定性

In [ ]:
print("加载数据...")

# 预测概率
pred_data = np.load(PRED_SOFTMAX_FILE)
pred_softmax = pred_data['pred_softmax_3d']
print(f"  预测概率: {pred_softmax.shape}")

# Ground Truth
gt_data = np.load(GT_3D_FILE)
gt_proba = gt_data['proba_labels']
region_mask = gt_data['region_mask_lr']
print(f"  Ground Truth: {gt_proba.shape}")

# MPRAGE底图
if DOWNSAMPLED_FILE.exists():
    downsampled_data = np.load(DOWNSAMPLED_FILE)
    data_lr = downsampled_data['data_lr']
    anatomy_img = data_lr[..., CHANNEL_MPRAGE]
    anatomy_name = "MPRAGE"
    print(f"  MPRAGE底图: {anatomy_img.shape}")
else:
    anatomy_img = np.zeros(pred_softmax.shape[:3])
    anatomy_name = "No Anatomy"

# ============================================================================
# 计算置信度
# ============================================================================

print("\n计算置信度与不确定性...")

top2_indices = np.argsort(pred_softmax, axis=-1)[..., -2:]
top1_class = top2_indices[..., 1]
top2_class = top2_indices[..., 0]

top2_probs = np.take_along_axis(pred_softmax, top2_indices, axis=-1)
p1 = top2_probs[..., 1]
p2 = top2_probs[..., 0]

# 边际差 (margin)
margin = p1 - p2  # (Z, X, Y)
alpha_map = np.clip(margin / TAU, 0, 1)
alpha_map[region_mask == 0] = 0

# ============================================================================
# 计算熵图（不确定性）
# ============================================================================

# H(p) = -Σ_k p_k log(p_k)
# 使用scipy.stats.entropy，axis=-1计算每个体素的熵
epsilon = 1e-10  # 避免log(0)
pred_softmax_safe = np.clip(pred_softmax, epsilon, 1.0)

# entropy使用自然对数，转换为bits需要除以log(2)
entropy_map = entropy(pred_softmax_safe.T, axis=0).T  # scipy要求axis=0，所以转置
entropy_map = entropy_map / np.log(2)  # 转为bits

# 应用ROI掩码
entropy_map_masked = entropy_map.copy()
entropy_map_masked[region_mask == 0] = 0

# 统计
entropy_roi = entropy_map[region_mask > 0]
margin_roi = margin[region_mask > 0]

print(f"  边际差 (Margin, ROI内):")
print(f"    Mean={margin_roi.mean():.4f}, Median={np.median(margin_roi):.4f}")
print(f"    Range=[{margin_roi.min():.4f}, {margin_roi.max():.4f}]")

print(f"  熵 (Entropy, ROI内):")
print(f"    Mean={entropy_roi.mean():.4f} bits, Median={np.median(entropy_roi):.4f} bits")
print(f"    Range=[{entropy_roi.min():.4f}, {entropy_roi.max():.4f}] bits")
print(f"    Max possible: {np.log2(N_CLASSES):.2f} bits (uniform distribution)")

# ============================================================================
# 自动选择切片
# ============================================================================

if AUTO_SELECT_SLICES:
    print("\n自动选择切片...")
    axis_size = pred_softmax.shape[0]
    slice_scores = []
    
    for z in range(axis_size):
        mask_slice = region_mask[z] > 0
        if mask_slice.sum() > 100:
            # 使用熵的标准差作为信息量度量
            entropy_std = entropy_map[z][mask_slice].std()
            mask_ratio = mask_slice.sum() / mask_slice.size
            slice_scores.append((z, entropy_std * mask_ratio))
    
    slice_scores.sort(key=lambda x: x[1], reverse=True)
    selected_slices = []
    for z, score in slice_scores:
        if len(selected_slices) == 0 or all(abs(z - s) >= 10 for s in selected_slices):
            selected_slices.append(z)
        if len(selected_slices) >= 3:
            break
    
    MANUAL_SLICES = sorted(selected_slices)
    print(f"  选择的切片: {MANUAL_SLICES}")

print("\n✓ 数据准备完成")

## 3. A+B并排可视化：概率叠加 vs 不确定性热图

In [ ]:
print("绘制A+B并排对比图...\n")

for slice_idx in MANUAL_SLICES:
    print(f"处理切片 {slice_idx}...")
    
    # 提取切片
    anatomy_slice = anatomy_img[slice_idx]
    top1_slice = top1_class[slice_idx]
    alpha_slice = alpha_map[slice_idx]
    mask_slice = region_mask[slice_idx]
    entropy_slice = entropy_map_masked[slice_idx]
    margin_slice = margin[slice_idx]
    
    # 创建2列图形 (A图 + B图)
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    
    # ========================================================================
    # 左图 (A): 概率叠加（Top-1 + 置信度）
    # ========================================================================
    
    # 背景：MPRAGE
    anatomy_norm = (anatomy_slice - anatomy_slice.min()) / (anatomy_slice.max() - anatomy_slice.min() + 1e-8)
    axes[0].imshow(anatomy_norm, cmap='gray', aspect='auto', interpolation='bilinear')
    
    # 叠加：Top-1类别 + 透明度
    top1_colored = cmap_freesurfer(top1_slice)
    top1_colored[..., 3] = alpha_slice
    axes[0].imshow(top1_colored, aspect='auto', interpolation='nearest')
    
    # ROI边界
    axes[0].contour(mask_slice, levels=[0.5], colors='cyan', 
                   linewidths=1.0, linestyles='--', alpha=0.5)
    
    axes[0].set_title(f'A: Probability Overlay\n'
                     f'{SUBJECT_ID} - Slice {slice_idx}\n'
                     f'Color=Top-1 ROI, Alpha=Confidence (p1-p2)',
                     fontsize=12, fontweight='bold', color='white')
    axes[0].axis('off')
    
    # ========================================================================
    # 右图 (B): 不确定性热图（熵图 + 边际差）
    # ========================================================================
    
    # 背景：边际差（灰度，归一化到[0,1]）
    margin_norm = (margin_slice - margin_slice[mask_slice > 0].min()) / \
                  (margin_slice[mask_slice > 0].max() - margin_slice[mask_slice > 0].min() + 1e-8)
    margin_norm[mask_slice == 0] = 0
    axes[1].imshow(margin_norm, cmap='gray', aspect='auto', interpolation='bilinear', 
                  vmin=0, vmax=1)
    
    # 叠加：熵图（伪彩色）
    # 使用'hot'或'jet' colormap，高熵=红色（难预测），低熵=蓝色（易预测）
    entropy_masked = np.ma.masked_where(mask_slice == 0, entropy_slice)
    
    im = axes[1].imshow(entropy_masked, cmap='hot', aspect='auto', 
                       interpolation='bilinear', alpha=0.7,
                       vmin=0, vmax=np.percentile(entropy_roi, 95))  # 95%分位数作为上限
    
    # 添加色条（熵值）
    cbar = plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_label('Entropy (bits)', rotation=270, labelpad=15, color='white', fontsize=10)
    cbar.ax.tick_params(colors='white', labelsize=9)
    
    # ROI边界
    axes[1].contour(mask_slice, levels=[0.5], colors='cyan', 
                   linewidths=1.0, linestyles='--', alpha=0.5)
    
    # 标注高不确定性区域（熵 > 75%分位数）
    high_entropy_threshold = np.percentile(entropy_roi, 75)
    high_entropy_mask = (entropy_slice > high_entropy_threshold) & (mask_slice > 0)
    
    if high_entropy_mask.sum() > 0:
        # 绘制高熵区域的等值线
        axes[1].contour(entropy_slice, levels=[high_entropy_threshold], 
                       colors='yellow', linewidths=2.0, alpha=0.8)
    
    axes[1].set_title(f'B: Uncertainty Heatmap\n'
                     f'Background=Margin (p1-p2), Overlay=Entropy H(p)\n'
                     f'High Entropy (yellow) = Hard Regions',
                     fontsize=12, fontweight='bold', color='white')
    axes[1].axis('off')
    
    # ========================================================================
    # 统一设置
    # ========================================================================
    
    plt.tight_layout()
    fig.patch.set_facecolor('black')
    
    # 保存
    save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_AB_slice{slice_idx:03d}_uncertainty.png'
    plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
    print(f"  ✓ 已保存: {save_path.name}")
    
    plt.show()
    plt.close()

print(f"\n✓ A+B并排对比完成！")

## 4. 纯不确定性热图（熵 + 边际差融合）

In [ ]:
print("绘制纯不确定性热图...\n")

for slice_idx in MANUAL_SLICES:
    print(f"处理切片 {slice_idx}...")
    
    anatomy_slice = anatomy_img[slice_idx]
    mask_slice = region_mask[slice_idx]
    entropy_slice = entropy_map_masked[slice_idx]
    margin_slice = margin[slice_idx]
    
    # 创建图形
    fig, ax = plt.subplots(1, 1, figsize=(14, 11))
    
    # 背景：MPRAGE灰度
    anatomy_norm = (anatomy_slice - anatomy_slice.min()) / (anatomy_slice.max() - anatomy_slice.min() + 1e-8)
    ax.imshow(anatomy_norm, cmap='gray', aspect='auto', interpolation='bilinear', alpha=0.3)
    
    # 主图：熵图（伪彩色）
    entropy_masked = np.ma.masked_where(mask_slice == 0, entropy_slice)
    im = ax.imshow(entropy_masked, cmap='hot', aspect='auto', 
                   interpolation='bilinear', alpha=0.8,
                   vmin=0, vmax=np.percentile(entropy_roi, 95))
    
    # 色条
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Entropy H(p) [bits]', rotation=270, labelpad=20, 
                   color='white', fontsize=11, fontweight='bold')
    cbar.ax.tick_params(colors='white', labelsize=10)
    
    # 边际差等值线（白色，显示置信度边界）
    margin_levels = [0.2, 0.4, 0.6]  # 不同置信度水平
    contours = ax.contour(margin_slice, levels=margin_levels, 
                         colors='white', linewidths=[1.0, 1.5, 2.0], 
                         alpha=0.6, linestyles='solid')
    ax.clabel(contours, inline=True, fontsize=8, fmt='Δp=%.1f', colors='white')
    
    # ROI边界
    ax.contour(mask_slice, levels=[0.5], colors='cyan', 
              linewidths=1.5, linestyles='--', alpha=0.7)
    
    # 标注最高不确定性的3个区域
    high_entropy_coords = np.argwhere(entropy_slice > np.percentile(entropy_roi, 95))
    if len(high_entropy_coords) > 0:
        # 随机选3个
        if len(high_entropy_coords) > 3:
            sample_idx = np.random.choice(len(high_entropy_coords), 3, replace=False)
            high_entropy_coords = high_entropy_coords[sample_idx]
        
        for coord in high_entropy_coords:
            x, y = coord
            h_val = entropy_slice[x, y]
            m_val = margin_slice[x, y]
            
            ax.plot(y, x, 'y*', markersize=12, markeredgewidth=1.5, markeredgecolor='black')
            ax.annotate(f'H={h_val:.2f}b\nΔp={m_val:.2f}',
                       xy=(y, x), xytext=(y+15, x-15),
                       fontsize=8, color='yellow', weight='bold',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='black', 
                                alpha=0.7, edgecolor='yellow'),
                       arrowprops=dict(arrowstyle='->', color='yellow', lw=1.5))
    
    # 标题
    ax.set_title(f'{SUBJECT_ID} - Uncertainty Heatmap\n'
                f'{SLICE_AXIS.capitalize()} Slice {slice_idx}\n'
                f'Hot Color = High Entropy (Hard to Predict)\n'
                f'White Contours = Margin Levels (p1-p2)',
                fontsize=13, fontweight='bold', pad=15, color='white')
    ax.axis('off')
    
    # 图例
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='white', linewidth=2, label='High Confidence (Δp>0.6)'),
        Line2D([0], [0], color='white', linewidth=1.5, label='Medium Confidence (Δp=0.4)'),
        Line2D([0], [0], color='white', linewidth=1, label='Low Confidence (Δp=0.2)'),
        Line2D([0], [0], color='cyan', linewidth=1.5, linestyle='--', label='ROI Boundary'),
        Line2D([0], [0], marker='*', color='w', markerfacecolor='yellow', 
               markersize=10, label='Highest Uncertainty', linestyle='None')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
             framealpha=0.85, edgecolor='white', facecolor='black', labelcolor='white')
    
    plt.tight_layout()
    fig.patch.set_facecolor('black')
    
    # 保存
    save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_uncertainty_slice{slice_idx:03d}.png'
    plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
    print(f"  ✓ 已保存: {save_path.name}")
    
    plt.show()
    plt.close()

print(f"\n✓ 不确定性热图完成！")

## 5. 统计分析：不确定性 vs 边际差

In [ ]:
print("统计分析：不确定性分布\n")

# 提取ROI内的值
entropy_roi = entropy_map[region_mask > 0]
margin_roi = margin[region_mask > 0]

# 创建散点图和直方图
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('black')

# ========================================================================
# 左上：熵的分布直方图
# ========================================================================
axes[0, 0].hist(entropy_roi, bins=50, color='red', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(entropy_roi.mean(), color='yellow', linestyle='--', linewidth=2, 
                   label=f'Mean={entropy_roi.mean():.2f}b')
axes[0, 0].axvline(np.median(entropy_roi), color='cyan', linestyle='--', linewidth=2,
                   label=f'Median={np.median(entropy_roi):.2f}b')
axes[0, 0].set_xlabel('Entropy H(p) [bits]', fontsize=11, color='white')
axes[0, 0].set_ylabel('Voxel Count', fontsize=11, color='white')
axes[0, 0].set_title('Entropy Distribution (ROI)', fontsize=12, fontweight='bold', color='white')
axes[0, 0].legend(fontsize=10, facecolor='black', edgecolor='white', labelcolor='white')
axes[0, 0].tick_params(colors='white')
axes[0, 0].set_facecolor('black')
for spine in axes[0, 0].spines.values():
    spine.set_edgecolor('white')

# ========================================================================
# 右上：边际差的分布直方图
# ========================================================================
axes[0, 1].hist(margin_roi, bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(margin_roi.mean(), color='yellow', linestyle='--', linewidth=2,
                   label=f'Mean={margin_roi.mean():.2f}')
axes[0, 1].axvline(np.median(margin_roi), color='cyan', linestyle='--', linewidth=2,
                   label=f'Median={np.median(margin_roi):.2f}')
axes[0, 1].set_xlabel('Margin (p1-p2)', fontsize=11, color='white')
axes[0, 1].set_ylabel('Voxel Count', fontsize=11, color='white')
axes[0, 1].set_title('Margin Distribution (ROI)', fontsize=12, fontweight='bold', color='white')
axes[0, 1].legend(fontsize=10, facecolor='black', edgecolor='white', labelcolor='white')
axes[0, 1].tick_params(colors='white')
axes[0, 1].set_facecolor('black')
for spine in axes[0, 1].spines.values():
    spine.set_edgecolor('white')

# ========================================================================
# 左下：熵 vs 边际差散点图
# ========================================================================
# 降采样显示（太多点会很慢）
sample_size = min(10000, len(entropy_roi))
sample_idx = np.random.choice(len(entropy_roi), sample_size, replace=False)

scatter = axes[1, 0].scatter(margin_roi[sample_idx], entropy_roi[sample_idx],
                            c=entropy_roi[sample_idx], cmap='hot', 
                            s=1, alpha=0.5)
axes[1, 0].set_xlabel('Margin (p1-p2)', fontsize=11, color='white')
axes[1, 0].set_ylabel('Entropy H(p) [bits]', fontsize=11, color='white')
axes[1, 0].set_title(f'Entropy vs Margin (n={sample_size:,} voxels)', 
                    fontsize=12, fontweight='bold', color='white')

# 计算相关系数
from scipy.stats import pearsonr, spearmanr
pearson_r, pearson_p = pearsonr(margin_roi, entropy_roi)
spearman_r, spearman_p = spearmanr(margin_roi, entropy_roi)



axes[1, 0].text(0.05, 0.95, 
                f'Pearson r={pearson_r:.3f} (p={pearson_p:.2e})\n'
                f'Spearman ρ={spearman_r:.3f} (p={spearman_p:.2e})',
                transform=axes[1, 0].transAxes, fontsize=10, 
                verticalalignment='top', color='yellow', weight='bold',
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.8, edgecolor='yellow'))

axes[1, 0].tick_params(colors='white')
axes[1, 0].set_facecolor('black')
for spine in axes[1, 0].spines.values():
    spine.set_edgecolor('white')

# ========================================================================
# 右下：累积分布函数（CDF）
# ========================================================================
entropy_sorted = np.sort(entropy_roi)
margin_sorted = np.sort(margin_roi)
cdf = np.arange(1, len(entropy_sorted) + 1) / len(entropy_sorted)

axes[1, 1].plot(entropy_sorted, cdf, color='red', linewidth=2, label='Entropy', alpha=0.8)
axes[1, 1].plot(margin_sorted, cdf, color='blue', linewidth=2, label='Margin', alpha=0.8)

# 标注分位数
for q in [0.25, 0.5, 0.75, 0.95]:
    axes[1, 1].axhline(q, color='white', linestyle=':', linewidth=1, alpha=0.5)
    axes[1, 1].text(0.01, q, f'{int(q*100)}%', color='white', fontsize=9,
                   verticalalignment='bottom')

axes[1, 1].set_xlabel('Value', fontsize=11, color='white')
axes[1, 1].set_ylabel('Cumulative Probability', fontsize=11, color='white')
axes[1, 1].set_title('Cumulative Distribution Functions', 
                    fontsize=12, fontweight='bold', color='white')
axes[1, 1].legend(fontsize=10, facecolor='black', edgecolor='white', labelcolor='white')
axes[1, 1].grid(True, alpha=0.3, color='white', linestyle=':')
axes[1, 1].tick_params(colors='white')
axes[1, 1].set_facecolor('black')
for spine in axes[1, 1].spines.values():
    spine.set_edgecolor('white')

# ========================================================================
# 总标题和保存
# ========================================================================
fig.suptitle(f'{SUBJECT_ID} - Uncertainty Analysis ({SPLIT_TYPE.upper()})',
            fontsize=14, fontweight='bold', color='white', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])

save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_uncertainty_analysis.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
print(f"✓ 已保存统计分析: {save_path.name}")

plt.show()
plt.close()

# 打印关键统计量
print("\n关键统计量:")
print(f"  熵 (Entropy):")
print(f"    Mean={entropy_roi.mean():.4f} bits, Std={entropy_roi.std():.4f} bits")
print(f"    P25={np.percentile(entropy_roi, 25):.4f}, P50={np.percentile(entropy_roi, 50):.4f}")
print(f"    P75={np.percentile(entropy_roi, 75):.4f}, P95={np.percentile(entropy_roi, 95):.4f}")
print(f"\n  边际差 (Margin):")
print(f"    Mean={margin_roi.mean():.4f}, Std={margin_roi.std():.4f}")
print(f"    P25={np.percentile(margin_roi, 25):.4f}, P50={np.percentile(margin_roi, 50):.4f}")
print(f"    P75={np.percentile(margin_roi, 75):.4f}, P95={np.percentile(margin_roi, 95):.4f}")
print(f"\n  相关性:")
print(f"    Pearson r = {pearson_r:.4f} (p={pearson_p:.2e})")
print(f"    Spearman ρ = {spearman_r:.4f} (p={spearman_p:.2e})")
print(f"\n  高不确定性体素 (Entropy > P75):")
high_entropy_count = (entropy_roi > np.percentile(entropy_roi, 75)).sum()
print(f"    Count={high_entropy_count:,} ({100*high_entropy_count/len(entropy_roi):.1f}%)")
print(f"\n✓ 统计分析完成！")

## 6. C图：可靠性图（Reliability Diagram，温度缩放前后对比）

In [ ]:
print("绘制可靠性图（Reliability Diagram）...\n")

from scipy.optimize import minimize
from sklearn.metrics import accuracy_score

# ============================================================================
# 准备数据：置信度 + Ground Truth标签
# ============================================================================

# 提取ROI内的预测概率和真实标签
roi_mask_flat = region_mask.flatten() > 0
pred_softmax_flat = pred_softmax.reshape(-1, N_CLASSES)[roi_mask_flat]  # (N_voxels, 102)

# Ground Truth: 使用概率标签的argmax作为真实类别
gt_proba_flat = gt_proba.reshape(-1, N_CLASSES)[roi_mask_flat]
gt_labels_flat = np.argmax(gt_proba_flat, axis=1)  # (N_voxels,)

# Top-1置信度和预测类别
confidences = np.max(pred_softmax_flat, axis=1)  # (N_voxels,)
predictions = np.argmax(pred_softmax_flat, axis=1)  # (N_voxels,)

print(f"ROI体素数: {len(confidences):,}")
print(f"预测类别范围: [{predictions.min()}, {predictions.max()}]")
print(f"置信度范围: [{confidences.min():.4f}, {confidences.max():.4f}]")

# ============================================================================
# 温度缩放（Temperature Scaling）
# ============================================================================

def temperature_scale(logits, T):
    """对logits应用温度缩放"""
    return logits / T

def compute_ece(confidences, predictions, labels, n_bins=15):
    """
    计算Expected Calibration Error (ECE)
    
    Args:
        confidences: 置信度 (N,)
        predictions: 预测类别 (N,)
        labels: 真实标签 (N,)
        n_bins: 分箱数
    
    Returns:
        ece: ECE值
        bin_data: 包含每个bin的统计信息的字典
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0.0
    bin_data = {
        'bin_centers': [],
        'accuracies': [],
        'confidences': [],
        'counts': []
    }
    
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        # 找到落在当前bin的样本
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.sum() / len(confidences)
        
        if prop_in_bin > 0:
            # 当前bin的准确率和平均置信度
            accuracy_in_bin = (predictions[in_bin] == labels[in_bin]).mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            
            # ECE累加
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            
            # 记录bin数据
            bin_data['bin_centers'].append((bin_lower + bin_upper) / 2)
            bin_data['accuracies'].append(accuracy_in_bin)
            bin_data['confidences'].append(avg_confidence_in_bin)
            bin_data['counts'].append(in_bin.sum())
    
    return ece, bin_data

def nll_loss(T, logits, labels):
    """
    负对数似然损失，用于优化温度参数
    
    Args:
        T: 温度参数 (scalar)
        logits: 模型输出的logits (N, C)
        labels: 真实标签 (N,)
    
    Returns:
        nll: 负对数似然
    """
    T = T[0] if isinstance(T, np.ndarray) else T
    
    # 应用温度缩放
    scaled_logits = logits / T
    
    # Softmax
    exp_logits = np.exp(scaled_logits - np.max(scaled_logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    
    # 负对数似然
    log_probs = np.log(probs[np.arange(len(labels)), labels] + 1e-12)
    nll = -np.mean(log_probs)
    
    return nll

print("\n优化温度参数T...")

# 将概率转为logits（近似）
# logits ≈ log(probs + epsilon)
epsilon = 1e-10
logits_flat = np.log(pred_softmax_flat + epsilon)

# 优化温度参数（使用验证集的NLL）
result = minimize(
    nll_loss,
    x0=[1.0],  # 初始温度=1.0
    args=(logits_flat, gt_labels_flat),
    method='L-BFGS-B',
    bounds=[(0.01, 10.0)],  # 温度范围
    options={'maxiter': 100}
)

T_optimal = result.x[0]
print(f"✓ 优化完成: T = {T_optimal:.4f}")

# ============================================================================
# 计算温度缩放前后的ECE和Reliability Diagram数据
# ============================================================================

# 温度缩放前
ece_before, bin_data_before = compute_ece(confidences, predictions, gt_labels_flat, n_bins=15)

# 温度缩放后
logits_scaled = temperature_scale(logits_flat, T_optimal)
exp_logits_scaled = np.exp(logits_scaled - np.max(logits_scaled, axis=1, keepdims=True))
probs_scaled = exp_logits_scaled / np.sum(exp_logits_scaled, axis=1, keepdims=True)
confidences_scaled = np.max(probs_scaled, axis=1)
predictions_scaled = np.argmax(probs_scaled, axis=1)

ece_after, bin_data_after = compute_ece(confidences_scaled, predictions_scaled, gt_labels_flat, n_bins=15)

print(f"\nECE改进:")
print(f"  Before: {ece_before:.4f}")
print(f"  After:  {ece_after:.4f}")
print(f"  Δ ECE:  {ece_before - ece_after:.4f} ({100*(ece_before - ece_after)/ece_before:.1f}% reduction)")

# 取绝对值（如果用户提到可能有负号）
ece_before = abs(ece_before)
ece_after = abs(ece_after)

# ============================================================================
# 绘制Reliability Diagram
# ============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 9))
fig.patch.set_facecolor('black')

# 完美校准线（对角线）
ax.plot([0, 1], [0, 1], 'w--', linewidth=2, alpha=0.5, label='Perfect Calibration')

# 温度缩放前的曲线
ax.plot(bin_data_before['confidences'], bin_data_before['accuracies'],
        'o-', color='red', linewidth=2.5, markersize=8,
        label=f'Before T-Scaling (ECE={ece_before:.4f})', alpha=0.85)

# 温度缩放后的曲线
ax.plot(bin_data_after['confidences'], bin_data_after['accuracies'],
        's-', color='cyan', linewidth=2.5, markersize=8,
        label=f'After T-Scaling (ECE={ece_after:.4f}, T={T_optimal:.3f})', alpha=0.85)

# 添加误差带（用柱状图显示样本密度）
ax2 = ax.twinx()
bin_counts = np.array(bin_data_before['counts'])
bin_counts_norm = bin_counts / bin_counts.max()  # 归一化到[0,1]
ax2.bar(bin_data_before['bin_centers'], bin_counts_norm,
        width=1/15, alpha=0.2, color='gray', label='Sample Density')
ax2.set_ylabel('Normalized Sample Density', fontsize=11, color='gray')
ax2.tick_params(axis='y', colors='gray', labelsize=9)
ax2.set_ylim([0, 1.2])

# 设置主坐标轴
ax.set_xlabel('Confidence', fontsize=12, fontweight='bold', color='white')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='white')
ax.set_title(f'{SUBJECT_ID} - Reliability Diagram\n'
            f'Temperature Scaling Calibration ({SPLIT_TYPE.upper()} Set)',
            fontsize=13, fontweight='bold', pad=15, color='white')

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, color='white', linestyle=':', linewidth=0.8)
ax.tick_params(colors='white', labelsize=10)
ax.set_facecolor('black')

for spine in ax.spines.values():
    spine.set_edgecolor('white')
for spine in ax2.spines.values():
    spine.set_edgecolor('gray')

# 图例
ax.legend(loc='upper left', fontsize=10, framealpha=0.9,
         edgecolor='white', facecolor='black', labelcolor='white')

# 添加统计信息文本框
info_text = (
    f"Calibration Improvement:\n"
    f"  ECE: {ece_before:.4f} → {ece_after:.4f}\n"
    f"  Reduction: {100*(ece_before - ece_after)/ece_before:.1f}%\n"
    f"  Temperature: T = {T_optimal:.3f}\n"
    f"  Voxels: {len(confidences):,}"
)
ax.text(0.98, 0.02, info_text,
       transform=ax.transAxes, fontsize=9,
       verticalalignment='bottom', horizontalalignment='right',
       color='yellow', weight='bold',
       bbox=dict(boxstyle='round,pad=0.5', facecolor='black',
                alpha=0.85, edgecolor='yellow', linewidth=1.5))

plt.tight_layout()

# 保存
save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_reliability_diagram.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
print(f"\n✓ 已保存可靠性图: {save_path.name}")

plt.show()
plt.close()

# ============================================================================
# 打印详细统计
# ============================================================================

print("\n详细统计:")
print(f"\n  置信度分箱 (n={len(bin_data_before['bin_centers'])} bins):")
print(f"  {'Bin':>5} {'Confidence':>12} {'Accuracy':>12} {'Count':>12} {'Gap':>12}")
print(f"  {'-'*5} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for i in range(len(bin_data_before['bin_centers'])):
    gap = abs(bin_data_before['confidences'][i] - bin_data_before['accuracies'][i])
    print(f"  {i+1:>5} {bin_data_before['confidences'][i]:>12.4f} "
          f"{bin_data_before['accuracies'][i]:>12.4f} "
          f"{bin_data_before['counts'][i]:>12,} {gap:>12.4f}")

print(f"\n✓ 可靠性图绘制完成！")

## 6B. C图改进版：硬标签 vs 软标签ECE对比（理论严谨版）

## 7. D图：风险-覆盖曲线（Risk-Coverage，选择性预测）

In [ ]:
print("绘制风险-覆盖曲线（Risk-Coverage Curve）...\n")

# ============================================================================
# 计算风险指标（NLL, Brier Score, Error Rate）
# ============================================================================

# 计算每个体素的风险指标
# 1. NLL (Negative Log-Likelihood)
nll_per_voxel = -np.log(pred_softmax_flat[np.arange(len(gt_labels_flat)), gt_labels_flat] + 1e-12)

# 2. Brier Score
brier_per_voxel = np.sum((pred_softmax_flat - gt_proba_flat) ** 2, axis=1)

# 3. Error Rate (0/1 loss)
error_per_voxel = (predictions != gt_labels_flat).astype(float)

print(f"风险指标统计:")
print(f"  NLL: Mean={nll_per_voxel.mean():.4f}, Std={nll_per_voxel.std():.4f}")
print(f"  Brier: Mean={brier_per_voxel.mean():.4f}, Std={brier_per_voxel.std():.4f}")
print(f"  Error Rate: {error_per_voxel.mean():.4f} ({100*error_per_voxel.mean():.2f}%)")

# ============================================================================
# 按不确定性排序，计算Risk-Coverage曲线
# ============================================================================

# 使用熵作为不确定性度量（也可以用margin的倒数）
uncertainty_scores = entropy_map.flatten()[roi_mask_flat]

# 按不确定性从低到高排序（越确定的越先保留）
sorted_indices = np.argsort(uncertainty_scores)

# 计算不同覆盖率下的风险
coverage_thresholds = np.linspace(0, 1, 101)  # 0%, 1%, 2%, ..., 100%
nll_at_coverage = []
brier_at_coverage = []
error_at_coverage = []
coverages = []

for cov in coverage_thresholds:
    n_keep = int(len(sorted_indices) * cov)
    
    if n_keep == 0:
        continue
    
    # 保留最确定的n_keep个体素
    keep_indices = sorted_indices[:n_keep]
    
    # 计算这些体素的平均风险
    nll_at_coverage.append(nll_per_voxel[keep_indices].mean())
    brier_at_coverage.append(brier_per_voxel[keep_indices].mean())
    error_at_coverage.append(error_per_voxel[keep_indices].mean())
    coverages.append(cov)

nll_at_coverage = np.array(nll_at_coverage)
brier_at_coverage = np.array(brier_at_coverage)
error_at_coverage = np.array(error_at_coverage)
coverages = np.array(coverages)

# 计算AURC (Area Under Risk-Coverage curve)
# AURC越小越好，表示在低覆盖率时风险下降快
aurc_nll = np.trapz(nll_at_coverage, coverages)
aurc_brier = np.trapz(brier_at_coverage, coverages)
aurc_error = np.trapz(error_at_coverage, coverages)

print(f"\nAURC (Area Under Risk-Coverage Curve):")
print(f"  AURC (NLL):   {aurc_nll:.4f}")
print(f"  AURC (Brier): {aurc_brier:.4f}")
print(f"  AURC (Error): {aurc_error:.4f}")

# ============================================================================
# 绘制Risk-Coverage曲线
# ============================================================================


fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('black')

risk_metrics = [
    (nll_at_coverage, 'NLL (Negative Log-Likelihood)', aurc_nll, 'orangered'),
    (brier_at_coverage, 'Brier Score', aurc_brier, 'dodgerblue'),
    (error_at_coverage, 'Error Rate', aurc_error, 'limegreen')
]

for idx, (risk, title, aurc, color) in enumerate(risk_metrics):
    ax = axes[idx]
    
    # main curve: Risk vs Coverage
    ax.plot(coverages * 100, risk, color=color, linewidth=2.5, alpha=0.9, label=f'AURC={aurc:.4f}')
    
    # fill area under curve
    ax.fill_between(coverages * 100, risk, alpha=0.2, color=color)
    
    # annotate key coverage points
    for cov_point in [0.5, 0.7, 0.9, 1.0]:
        if cov_point <= coverages.max():
            cov_idx = np.argmin(np.abs(coverages - cov_point))
            risk_val = risk[cov_idx]
            ax.plot(cov_point * 100, risk_val, 'o', color='yellow', markersize=8, 
                   markeredgecolor='white', markeredgewidth=1.5, zorder=5)
            ax.annotate(f'{int(cov_point*100)}%: {risk_val:.3f}',
                       xy=(cov_point * 100, risk_val),
                       xytext=(cov_point * 100 + 2, risk_val + (risk.max() - risk.min()) * 0.05),
                       fontsize=8, color='yellow', weight='bold',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='black', 
                                alpha=0.7, edgecolor='yellow'),
                       arrowprops=dict(arrowstyle='->', color='yellow', lw=1))
    
    # axes labels/titles
    ax.set_xlabel('Coverage (% of voxels predicted)', fontsize=11, fontweight='bold', color='white')
    ax.set_ylabel(f'Risk ({title})', fontsize=11, fontweight='bold', color='white')
    ax.set_title(f'{title}\nSelective Prediction Performance',
                fontsize=12, fontweight='bold', pad=10, color='white')
    
    ax.set_xlim([0, 100])
    ax.set_ylim([0, risk.max() * 1.1])
    ax.grid(True, alpha=0.3, color='white', linestyle=':', linewidth=0.8)
    ax.tick_params(colors='white', labelsize=9)
    ax.set_facecolor('black')
    
    for spine in ax.spines.values():
        spine.set_edgecolor('white')
    
    # legend
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9,
             edgecolor='white', facecolor='black', labelcolor='white')
    
    # info text (translated)
    info_text = (
        f"Coverage = fraction of voxels kept\n"
        f"Risk = {title}\n"
        f"Lower AURC = Better"
    )
    ax.text(0.02, 0.98, info_text,
           transform=ax.transAxes, fontsize=8,
           verticalalignment='top', horizontalalignment='left',
           color='cyan', weight='bold',
           bbox=dict(boxstyle='round,pad=0.4', facecolor='black',
                    alpha=0.8, edgecolor='cyan'))

# figure suptitle (translated)
fig.suptitle(f'{SUBJECT_ID} - Risk-Coverage Curves ({SPLIT_TYPE.upper()} Set)\n'
            f'Selective Prediction: Performance vs Coverage Trade-off',
            fontsize=14, fontweight='bold', color='white', y=1.02)

plt.tight_layout()

# save (filename unchanged)
save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_risk_coverage.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
print(f"\n✓ 已保存风险-覆盖曲线: {save_path.name}")

plt.show()
plt.close()

# ============================================================================
# 打印关键统计
# ============================================================================

print("\n关键统计:")
print(f"\n  选择性预测性能（保留最确定的50%体素）:")
cov_50_idx = np.argmin(np.abs(coverages - 0.5))
print(f"    NLL:   {nll_at_coverage[cov_50_idx]:.4f} (vs 全部: {nll_per_voxel.mean():.4f})")
print(f"    Brier: {brier_at_coverage[cov_50_idx]:.4f} (vs 全部: {brier_per_voxel.mean():.4f})")
print(f"    Error: {error_at_coverage[cov_50_idx]:.4f} (vs 全部: {error_per_voxel.mean():.4f})")

print(f"\n  选择性预测性能（保留最确定的70%体素）:")
cov_70_idx = np.argmin(np.abs(coverages - 0.7))
print(f"    NLL:   {nll_at_coverage[cov_70_idx]:.4f} (vs 全部: {nll_per_voxel.mean():.4f})")
print(f"    Brier: {brier_at_coverage[cov_70_idx]:.4f} (vs 全部: {brier_per_voxel.mean():.4f})")
print(f"    Error: {error_at_coverage[cov_70_idx]:.4f} (vs 全部: {error_per_voxel.mean():.4f})")

print(f"\n  选择性预测性能（保留最确定的90%体素）:")
cov_90_idx = np.argmin(np.abs(coverages - 0.9))
print(f"    NLL:   {nll_at_coverage[cov_90_idx]:.4f} (vs 全部: {nll_per_voxel.mean():.4f})")
print(f"    Brier: {brier_at_coverage[cov_90_idx]:.4f} (vs 全部: {brier_per_voxel.mean():.4f})")
print(f"    Error: {error_at_coverage[cov_90_idx]:.4f} (vs 全部: {error_per_voxel.mean():.4f})")

print(f"\n✓ 风险-覆盖曲线绘制完成！")

## 8. E图：软混淆邻接热图（Soft Confusion Matrix，解释"谁和谁混"）

In [ ]:
print("绘制软混淆邻接热图（Soft Confusion Matrix with Adjacency Mask）...\n")

from scipy.ndimage import binary_dilation, generate_binary_structure

# ============================================================================
# 计算软混淆矩阵（Soft Confusion Matrix）
# ============================================================================

print("计算软混淆矩阵...")

# 初始化软混淆矩阵 (102 x 102)
soft_confusion = np.zeros((N_CLASSES, N_CLASSES))

# 对每个体素，累加预测概率到对应的GT类别行
for i in range(len(gt_labels_flat)):
    gt_class = gt_labels_flat[i]
    pred_probs = pred_softmax_flat[i]
    
    # 累加：soft_confusion[gt_class, :] += pred_probs
    soft_confusion[gt_class, :] += pred_probs

# 归一化到GT质量（每行和为1，表示该GT类别的预测概率分布）
row_sums = soft_confusion.sum(axis=1, keepdims=True)
soft_confusion_norm = np.divide(soft_confusion, row_sums, 
                                 out=np.zeros_like(soft_confusion), 
                                 where=row_sums > 0)

print(f"  软混淆矩阵形状: {soft_confusion_norm.shape}")
print(f"  对角线均值: {np.diag(soft_confusion_norm).mean():.4f}")
print(f"  非对角线均值: {soft_confusion_norm[~np.eye(N_CLASSES, dtype=bool)].mean():.4f}")

# ============================================================================
# 自动推断空间邻接关系（从region_mask_lr）
# ============================================================================

print("\n推断空间邻接关系...")

# 为每个ROI创建二值掩膜
roi_masks = {}
for label_idx in range(N_CLASSES):
    roi_masks[label_idx] = (region_mask == label_idx)

# 定义26-连通邻域结构（3D）
struct = generate_binary_structure(3, 3)  # 26-connectivity

# 计算邻接矩阵
adjacency_matrix = np.zeros((N_CLASSES, N_CLASSES), dtype=bool)

for i in range(N_CLASSES):
    if not roi_masks[i].any():
        continue
    
    # 膨胀ROI i的掩膜
    dilated_i = binary_dilation(roi_masks[i], structure=struct, iterations=1)
    
    for j in range(i + 1, N_CLASSES):  # 只计算上三角，因为邻接是对称的
        if not roi_masks[j].any():
            continue
        
        # 检查ROI j是否与膨胀后的ROI i重叠
        if np.any(dilated_i & roi_masks[j]):
            adjacency_matrix[i, j] = True
            adjacency_matrix[j, i] = True  # 对称

# 对角线设为True（自己和自己邻接）
np.fill_diagonal(adjacency_matrix, True)

n_adjacent_pairs = (adjacency_matrix.sum() - N_CLASSES) // 2  # 减去对角线，除以2（对称）
print(f"  发现 {n_adjacent_pairs} 对邻接ROI对")

# ============================================================================
# 应用邻接掩膜
# ============================================================================

# 只保留邻接的ROI对
soft_confusion_adjacent = soft_confusion_norm.copy()
soft_confusion_adjacent[~adjacency_matrix] = 0  # 非邻接的置零

# ============================================================================
# 找出Top-10混淆对（非对角线，邻接的）
# ============================================================================

print("\n识别Top-10混淆对...")

confusion_pairs = []
for i in range(N_CLASSES):
    for j in range(i + 1, N_CLASSES):  # 只看上三角
        if adjacency_matrix[i, j]:
            # 双向混淆：i->j 和 j->i 的平均值
            confusion_ij = soft_confusion_norm[i, j]
            confusion_ji = soft_confusion_norm[j, i]
            avg_confusion = (confusion_ij + confusion_ji) / 2
            
            if avg_confusion > 0:
                confusion_pairs.append({
                    'roi1': i,
                    'roi2': j,
                    'roi1_name': LABEL_NAMES.get(i, f'ROI_{i}'),
                    'roi2_name': LABEL_NAMES.get(j, f'ROI_{j}'),
                    'confusion_ij': confusion_ij,
                    'confusion_ji': confusion_ji,
                    'avg_confusion': avg_confusion
                })

# 按平均混淆度降序排序
confusion_pairs.sort(key=lambda x: x['avg_confusion'], reverse=True)
top10_pairs = confusion_pairs[:10]

print(f"  Top-10混淆对:")
for idx, pair in enumerate(top10_pairs):
    print(f"    {idx+1}. {pair['roi1_name']} <-> {pair['roi2_name']}: "
          f"{pair['avg_confusion']:.4f} ({pair['confusion_ij']:.4f}/{pair['confusion_ji']:.4f})")

# ============================================================================
# 绘制：热图 + Top-10条形图
# ============================================================================

fig = plt.figure(figsize=(20, 8))
fig.patch.set_facecolor('black')

# 创建GridSpec布局
from matplotlib.gridspec import GridSpec
gs = GridSpec(1, 2, width_ratios=[2, 1], wspace=0.3)

# ========================================================================
# 左图：软混淆邻接热图
# ========================================================================

ax_heatmap = fig.add_subplot(gs[0])

# 使用邻接掩膜后的混淆矩阵
# 创建masked array以区分邻接和非邻接
soft_confusion_masked = np.ma.masked_where(~adjacency_matrix, soft_confusion_norm)

# 绘制热图
im = ax_heatmap.imshow(soft_confusion_masked, cmap='hot', aspect='auto', 
                       interpolation='nearest', vmin=0, vmax=0.5)

# 添加色条
cbar = plt.colorbar(im, ax=ax_heatmap, fraction=0.046, pad=0.04)
cbar.set_label('Confusion Probability\n(normalized by GT mass)', 
              rotation=270, labelpad=20, color='white', fontsize=10, fontweight='bold')
cbar.ax.tick_params(colors='white', labelsize=9)

# 标注Top-10混淆对
for idx, pair in enumerate(top10_pairs[:5]):  # 只标注前5个，避免太拥挤
    i, j = pair['roi1'], pair['roi2']
    
    # 在热图上画圈
    ax_heatmap.plot(j, i, 'o', markersize=10, markerfacecolor='none', 
                   markeredgecolor='cyan', markeredgewidth=2, alpha=0.8)
    ax_heatmap.plot(i, j, 'o', markersize=10, markerfacecolor='none', 
                   markeredgecolor='cyan', markeredgewidth=2, alpha=0.8)
    
    # 标注数字
    ax_heatmap.text(j, i, f'{idx+1}', fontsize=8, color='cyan', 
                   weight='bold', ha='center', va='center')
    ax_heatmap.text(i, j, f'{idx+1}', fontsize=8, color='cyan', 
                   weight='bold', ha='center', va='center')

# 坐标轴设置
ax_heatmap.set_xlabel('Predicted ROI', fontsize=11, fontweight='bold', color='white')
ax_heatmap.set_ylabel('Ground Truth ROI', fontsize=11, fontweight='bold', color='white')
ax_heatmap.set_title(f'Soft Confusion Matrix (Adjacency-Masked)\n'
                    f'{SUBJECT_ID} - {SPLIT_TYPE.upper()} Set\n'
                    f'Only Anatomically Adjacent ROI Pairs Shown',
                    fontsize=12, fontweight='bold', pad=15, color='white')

ax_heatmap.tick_params(colors='white', labelsize=8)
ax_heatmap.set_facecolor('black')

for spine in ax_heatmap.spines.values():
    spine.set_edgecolor('white')

# 添加说明文本
info_text = (
    f"Gray = non-adjacent ROI pairs\n"
    f"Warm colors = adjacent pairs with confusion\n"
    f"Cyan circles = Top-5 confused pairs"
)
ax_heatmap.text(0.02, 0.98, info_text,
               transform=ax_heatmap.transAxes, fontsize=9,
               verticalalignment='top', horizontalalignment='left',
               color='yellow', weight='bold',
               bbox=dict(boxstyle='round,pad=0.4', facecolor='black',
                        alpha=0.8, edgecolor='yellow'))

# ========================================================================
# 右图：Top-10混淆对条形图
# ========================================================================

ax_bar = fig.add_subplot(gs[1])

# 准备数据
pair_labels = []
pair_confusions = []

for idx, pair in enumerate(top10_pairs):
    label = f"{pair['roi1_name'][:15]}\n↔\n{pair['roi2_name'][:15]}"
    pair_labels.append(label)
    pair_confusions.append(pair['avg_confusion'] * 100)  # 转为百分比

# 绘制水平条形图
y_pos = np.arange(len(pair_labels))
bars = ax_bar.barh(y_pos, pair_confusions, color='dodgerblue', alpha=0.8, edgecolor='white')

# 渐变颜色
for i, bar in enumerate(bars):
    bar.set_color(plt.cm.hot(1 - i / len(bars)))

# 标注数值
for i, (y, val) in enumerate(zip(y_pos, pair_confusions)):
    ax_bar.text(val + 0.2, y, f'{val:.2f}%', 
               va='center', ha='left', fontsize=9, color='white', weight='bold')

# 坐标轴设置
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(pair_labels, fontsize=8)
ax_bar.set_xlabel('Average Confusion (%)', fontsize=10, fontweight='bold', color='white')
ax_bar.set_title('Top-10 Confused ROI Pairs\n(Anatomically Adjacent)',
                fontsize=11, fontweight='bold', pad=10, color='white')

ax_bar.invert_yaxis()  # 最高的在上面
ax_bar.set_xlim([0, max(pair_confusions) * 1.15])
ax_bar.grid(axis='x', alpha=0.3, color='white', linestyle=':', linewidth=0.8)
ax_bar.tick_params(colors='white', labelsize=9)
ax_bar.set_facecolor('black')

for spine in ax_bar.spines.values():
    spine.set_edgecolor('white')

# ========================================================================
# 保存
# ========================================================================

plt.tight_layout()

save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_soft_confusion_adjacency.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
print(f"\n✓ 已保存软混淆邻接热图: {save_path.name}")

plt.show()
plt.close()

# ============================================================================
# 导出CSV文件
# ============================================================================

print("\n导出软混淆矩阵到CSV...")

# 完整软混淆矩阵
df_soft_confusion = pd.DataFrame(soft_confusion_norm, 
                                 index=[LABEL_NAMES.get(i, f'ROI_{i}') for i in range(N_CLASSES)],
                                 columns=[LABEL_NAMES.get(i, f'ROI_{i}') for i in range(N_CLASSES)])
csv_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_soft_confusion_full.csv'
df_soft_confusion.to_csv(csv_path)
print(f"  ✓ 已保存完整软混淆矩阵: {csv_path.name}")

# Top-10混淆对
df_top10 = pd.DataFrame(top10_pairs)
csv_path_top10 = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_soft_confusion_top10.csv'
df_top10.to_csv(csv_path_top10, index=False)
print(f"  ✓ 已保存Top-10混淆对: {csv_path_top10.name}")

print(f"\n✓ 软混淆邻接热图绘制完成！")

---

## 总结

### 完成的可视化：

1. **A+B并排对比**
   - **A图**: 概率叠加（Top-1类别 + 置信度透明度）
   - **B图**: 不确定性热图（熵伪彩 + 边际差灰度）

2. **纯不确定性热图**
   - 熵图（'hot' colormap）
   - 边际差等值线（白色，显示置信度水平）
   - 自动标注最高不确定性区域

3. **统计分析**
   - 熵和边际差的分布直方图
   - 熵 vs 边际差散点图（含相关系数）
   - 累积分布函数（CDF）

4. **C图：可靠性图（Reliability Diagram）**
   - 温度缩放前后对比曲线
   - ECE改进数值（例如：0.0401 → 0.0292）
   - 温度参数T标注在图例中
   - 样本密度柱状图（灰色背景）

5. **D图：风险-覆盖曲线（Risk-Coverage）**
   - NLL、Brier Score、Error Rate三种风险指标
   - 选择性预测性能分析
   - AURC值计算
   - 关键覆盖率点标注（50%、70%、90%、100%）

6. **E图：软混淆邻接热图（Soft Confusion Matrix）**
   - 空间邻接ROI对混淆分析
   - 自动推断26-连通邻接关系
   - Top-10混淆对条形图
   - CSV导出（完整矩阵 + Top-10对）

7. **F图：部分体素指数（PV Index）边界条带评估**
   - 边界条带检测（±1 voxel）
   - Margin/Entropy分布可视化
   - Soft Dice计算（tolerance=1 voxel）
   - 逐ROI统计（CSV导出）

### 关键指标：

- **熵 H(p) = -Σ_k p_k log(p_k)**
  - 单位：bits
  - 范围：[0, log₂(102)] ≈ [0, 6.67] bits
  - 含义：高熵 = 概率分布均匀 = 模型不确定

- **边际差 (Margin) = p1 - p2**
  - 范围：[0, 1]
  - 含义：低边际差 = Top-1和Top-2接近 = 竞争激烈

- **ECE (Expected Calibration Error)**
  - 范围：[0, 1]
  - 含义：预测置信度与实际准确率的加权平均偏差
  - 计算：ECE = Σ_b (|conf_b - acc_b|) × (n_b / N)

- **温度缩放 (Temperature Scaling)**
  - logits' = logits / T
  - T > 1: 软化概率分布（降低过度自信）
  - T < 1: 锐化概率分布（增加自信）
  - T = 1: 无缩放（原始预测）

- **AURC (Area Under Risk-Coverage Curve)**
  - 范围：[0, ∞)，越小越好
  - 含义：选择性预测的综合性能
  - 用途：评估"只预测最确定体素"时的风险降低

- **软混淆矩阵 (Soft Confusion Matrix)**
  - 归一化：按GT质量归一化（行和=1）
  - 邻接掩膜：只保留空间相邻的ROI对
  - 含义：区分"合理边界混合"vs"真实误分类"

- **PV指数 (Partial Volume Index)**
  - 边界条带：外扩1体素 + 内缩1体素
  - Margin中位数：边界处的平均置信度
  - Entropy P95：边界处的高不确定性阈值
  - Soft Dice (tolerance=1)：考虑边界误差的Dice系数

### 可视化策略：

- **定性分析（A、B图）**: 展示"模型在哪里预测什么"和"哪里不确定"
- **定量校准（C图）**: 证明概率是可信的（温度缩放改进ECE）
- **选择性预测（D图）**: 展示"临床只保留高置信部分"时的性能
- **混淆分析（E图）**: 拥抱不确定性，区分边界效应vs真实错误
- **边界评估（F图）**: 量化边界处的软混合，证明概率化训练的价值

### 不确定性来源：

- **部分体素效应**：边界体素包含多个组织
- **配准误差**：解剖对齐不完美
- **类别本体难度**：某些脑区天然难以区分（如核团边界）
- **标签噪声**：Ground truth本身的不确定性

### 临床意义：

- **选择性预测**：可以只保留最确定的70%体素，大幅降低错误率
- **概率校准**：温度缩放后的概率更可信，可用于临床决策
- **混淆分析**：识别哪些ROI对容易混淆，指导后续改进方向
- **边界容忍**：Soft Dice (tolerance=1)比硬Dice更能体现概率化训练的优势

### 使用方法：

1. 修改第1个单元格的 `SUBJECT_ID` 和 `SPLIT_TYPE`
2. 运行所有单元格
3. 结果保存在 `{FOLD_DIR}/figs/confidence_overlay/`
4. CSV文件保存在同一目录下：
   - `soft_confusion_full.csv` - 完整软混淆矩阵
   - `soft_confusion_top10.csv` - Top-10混淆对
   - `pv_boundary_stats.csv` - 边界条带统计

---

**版本**: v5.0  
**更新**: 添加F图（部分体素指数边界条带评估）  
**日期**: 2025-01-14

---

## 总结

### 完成的可视化：

1. **A+B并排对比**
   - **A图**: 概率叠加（Top-1类别 + 置信度透明度）
   - **B图**: 不确定性热图（熵伪彩 + 边际差灰度）

2. **纯不确定性热图**
   - 熵图（'hot' colormap）
   - 边际差等值线（白色，显示置信度水平）
   - 自动标注最高不确定性区域

3. **统计分析**
   - 熵和边际差的分布直方图
   - 熵 vs 边际差散点图（含相关系数）
   - 累积分布函数（CDF）

4. **C图：可靠性图（Reliability Diagram）**
   - 温度缩放前后对比曲线
   - ECE改进数值（例如：0.0401 → 0.0292）
   - 温度参数T标注在图例中
   - 样本密度柱状图（灰色背景）

5. **D图：风险-覆盖曲线（Risk-Coverage）**
   - NLL、Brier Score、Error Rate三种风险指标
   - 选择性预测性能分析
   - AURC值计算
   - 关键覆盖率点标注（50%、70%、90%、100%）

6. **E图：软混淆邻接热图（Soft Confusion Matrix）**
   - 空间邻接ROI对混淆分析
   - 自动推断26-连通邻接关系
   - Top-10混淆对条形图
   - CSV导出（完整矩阵 + Top-10对）

### 关键指标：

- **熵 H(p) = -Σ_k p_k log(p_k)**
  - 单位：bits
  - 范围：[0, log₂(102)] ≈ [0, 6.67] bits
  - 含义：高熵 = 概率分布均匀 = 模型不确定

- **边际差 (Margin) = p1 - p2**
  - 范围：[0, 1]
  - 含义：低边际差 = Top-1和Top-2接近 = 竞争激烈

- **ECE (Expected Calibration Error)**
  - 范围：[0, 1]
  - 含义：预测置信度与实际准确率的加权平均偏差
  - 计算：ECE = Σ_b (|conf_b - acc_b|) × (n_b / N)

- **温度缩放 (Temperature Scaling)**
  - logits' = logits / T
  - T > 1: 软化概率分布（降低过度自信）
  - T < 1: 锐化概率分布（增加自信）
  - T = 1: 无缩放（原始预测）

- **AURC (Area Under Risk-Coverage Curve)**
  - 范围：[0, ∞)，越小越好
  - 含义：选择性预测的综合性能
  - 用途：评估"只预测最确定体素"时的风险降低

- **软混淆矩阵 (Soft Confusion Matrix)**
  - 归一化：按GT质量归一化（行和=1）
  - 邻接掩膜：只保留空间相邻的ROI对
  - 含义：区分"合理边界混合"vs"真实误分类"

### 可视化策略：

- **定性分析（A、B图）**: 展示"模型在哪里预测什么"和"哪里不确定"
- **定量校准（C图）**: 证明概率是可信的（温度缩放改进ECE）
- **选择性预测（D图）**: 展示"临床只保留高置信部分"时的性能
- **混淆分析（E图）**: 拥抱不确定性，区分边界效应vs真实错误

### 不确定性来源：

- **部分体素效应**：边界体素包含多个组织
- **配准误差**：解剖对齐不完美
- **类别本体难度**：某些脑区天然难以区分（如核团边界）
- **标签噪声**：Ground truth本身的不确定性

### 临床意义：

- **选择性预测**：可以只保留最确定的70%体素，大幅降低错误率
- **概率校准**：温度缩放后的概率更可信，可用于临床决策
- **混淆分析**：识别哪些ROI对容易混淆，指导后续改进方向

### 使用方法：

1. 修改第1个单元格的 `SUBJECT_ID` 和 `SPLIT_TYPE`
2. 运行所有单元格
3. 结果保存在 `{FOLD_DIR}/figs/confidence_overlay/`
4. CSV文件保存在同一目录下

---

**版本**: v4.0  
**更新**: 添加D图（风险-覆盖曲线）+ E图（软混淆邻接热图）  
**日期**: 2025-01-14

In [ ]:
print("绘制部分体素指数（PV Index）边界条带评估...\n")

from scipy.ndimage import binary_erosion, binary_dilation, generate_binary_structure

# ============================================================================
# 检测所有ROI的边界条带
# ============================================================================

print("检测ROI边界条带...")

# 定义3D连通结构（6-连通，只考虑面相邻）
struct_6 = generate_binary_structure(3, 1)  # 6-connectivity

# 存储所有ROI的边界条带
boundary_bands = {}
roi_stats = []

for roi_idx in range(1, N_CLASSES):  # 跳过背景（0）
    roi_mask = (region_mask == roi_idx)
    
    if not roi_mask.any():
        continue
    
    # 外扩1个体素（膨胀）
    dilated = binary_dilation(roi_mask, structure=struct_6, iterations=1)
    
    # 内缩1个体素（腐蚀）
    eroded = binary_erosion(roi_mask, structure=struct_6, iterations=1)
    
    # 边界条带 = (膨胀 - 腐蚀)
    boundary_band = dilated & ~eroded
    
    # 排除原ROI内部（只保留边界附近）
    # 实际上，我们要的是：外扩层 + 内缩层
    outer_layer = dilated & ~roi_mask  # 外扩层
    inner_layer = roi_mask & ~eroded   # 内缩层
    boundary_band = outer_layer | inner_layer
    
    if boundary_band.sum() > 0:
        boundary_bands[roi_idx] = boundary_band

print(f"  检测到 {len(boundary_bands)} 个ROI的边界条带")

# ============================================================================
# 统计边界条带内的PV指标
# ============================================================================

print("\n计算边界条带统计...")

for roi_idx, band_mask in boundary_bands.items():
    roi_name = LABEL_NAMES.get(roi_idx, f'ROI_{roi_idx}')
    
    # 提取边界条带内的值
    band_mask_flat = band_mask.flatten()
    
    # 边界条带内的预测概率
    pred_softmax_band = pred_softmax.reshape(-1, N_CLASSES)[band_mask_flat]
    
    # 边界条带内的GT概率
    gt_proba_band = gt_proba.reshape(-1, N_CLASSES)[band_mask_flat]
    
    # 边界条带内的Top-2概率和类别
    top2_probs_band = np.sort(pred_softmax_band, axis=1)[:, -2:]
    top2_classes_band = np.argsort(pred_softmax_band, axis=1)[:, -2:]
    
    p1_band = top2_probs_band[:, 1]
    p2_band = top2_probs_band[:, 0]
    c1_band = top2_classes_band[:, 1]
    c2_band = top2_classes_band[:, 0]
    
    # 边际差 (margin)
    margin_band = p1_band - p2_band
    
    # 熵
    entropy_band = entropy(pred_softmax_band.T, axis=0).T / np.log(2)
    
    # PV指数：p1/(p1+p2) - 衡量主类的相对优势
    pv_index = p1_band / (p1_band + p2_band + 1e-10)
    
    # 不确定性指数：1 - margin
    uncertainty_index = 1 - margin_band
    
    # 软Dice（考虑边界tolerance=1 voxel）
    # 计算当前ROI的预测概率和GT概率的重叠
    pred_prob_roi = pred_softmax_band[:, roi_idx]
    gt_prob_roi = gt_proba_band[:, roi_idx]
    
    soft_dice_numerator = 2 * np.sum(pred_prob_roi * gt_prob_roi)
    soft_dice_denominator = np.sum(pred_prob_roi) + np.sum(gt_prob_roi)
    soft_dice = soft_dice_numerator / (soft_dice_denominator + 1e-10)
    
    # 统计
    stats = {
        'roi_idx': roi_idx,
        'roi_name': roi_name,
        'n_voxels': band_mask.sum(),
        'margin_median': np.median(margin_band),
        'margin_mean': np.mean(margin_band),
        'margin_std': np.std(margin_band),
        'entropy_p95': np.percentile(entropy_band, 95),
        'entropy_median': np.median(entropy_band),
        'pv_index_median': np.median(pv_index),
        'pv_index_mean': np.mean(pv_index),
        'uncertainty_median': np.median(uncertainty_index),
        'soft_dice': soft_dice
    }
    
    roi_stats.append(stats)

# 转为DataFrame
df_roi_stats = pd.DataFrame(roi_stats)

# 排序：按边界体素数降序
df_roi_stats = df_roi_stats.sort_values('n_voxels', ascending=False)

print(f"  完成 {len(df_roi_stats)} 个ROI的统计")

# 打印Top-10 ROI的统计
print("\n边界条带统计（Top-10 ROI by voxel count）:")
print(f"  {'ROI':<25} {'N_voxels':>10} {'Margin':>10} {'Entropy':>10} {'PV_Index':>10} {'Soft_Dice':>10}")
print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
for idx, row in df_roi_stats.head(10).iterrows():
    print(f"  {row['roi_name'][:25]:<25} {row['n_voxels']:>10,} "
          f"{row['margin_median']:>10.3f} {row['entropy_p95']:>10.3f} "
          f"{row['pv_index_median']:>10.3f} {row['soft_dice']:>10.3f}")

# ============================================================================
# 可视化：边界条带叠加 + 统计图
# ============================================================================

print("\n绘制边界条带可视化...")

fig = plt.figure(figsize=(22, 12))
fig.patch.set_facecolor('black')

from matplotlib.gridspec import GridSpec
gs = GridSpec(2, 3, height_ratios=[1.2, 1], wspace=0.3, hspace=0.3)

# ========================================================================
# 第一行：选择一个代表性切片，展示边界条带
# ========================================================================

# 选择边界体素最多的切片
boundary_all = np.zeros_like(region_mask, dtype=bool)
for band_mask in boundary_bands.values():
    boundary_all |= band_mask

boundary_counts_per_slice = boundary_all.sum(axis=(1, 2))
best_slice_idx = np.argmax(boundary_counts_per_slice)

print(f"  选择切片 {best_slice_idx}（边界体素最多）")

# 准备切片数据
anatomy_slice = anatomy_img[best_slice_idx]
boundary_slice = boundary_all[best_slice_idx]
margin_slice = margin[best_slice_idx]
entropy_slice = entropy_map[best_slice_idx]

# 左图：边界条带叠加在解剖图上
ax1 = fig.add_subplot(gs[0, 0])

anatomy_norm = (anatomy_slice - anatomy_slice.min()) / (anatomy_slice.max() - anatomy_slice.min() + 1e-8)
ax1.imshow(anatomy_norm, cmap='gray', aspect='auto', interpolation='bilinear')

# 叠加边界条带（红色高亮）
boundary_colored = np.zeros((*boundary_slice.shape, 4))
boundary_colored[boundary_slice, :] = [1, 0, 0, 0.6]  # 红色，60%透明度
ax1.imshow(boundary_colored, aspect='auto', interpolation='nearest')

ax1.set_title(f'Boundary Band Visualization\nSlice {best_slice_idx}\n'
             f'Red = Boundary Voxels (±1 voxel from ROI edges)',
             fontsize=11, fontweight='bold', color='white', pad=10)
ax1.axis('off')

# 中图：边界条带内的Margin分布
ax2 = fig.add_subplot(gs[0, 1])

margin_boundary = np.ma.masked_where(~boundary_slice, margin_slice)
im2 = ax2.imshow(margin_boundary, cmap='RdYlGn', aspect='auto', 
                interpolation='nearest', vmin=0, vmax=1)

cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label('Margin (p1-p2)', rotation=270, labelpad=15, color='white', fontsize=9)
cbar2.ax.tick_params(colors='white', labelsize=8)

ax2.set_title(f'Margin at Boundary\n'
             f'Green = High Confidence\nRed = Low Confidence',
             fontsize=11, fontweight='bold', color='white', pad=10)
ax2.axis('off')

# 右图：边界条带内的Entropy分布
ax3 = fig.add_subplot(gs[0, 2])

entropy_boundary = np.ma.masked_where(~boundary_slice, entropy_slice)
im3 = ax3.imshow(entropy_boundary, cmap='hot', aspect='auto', 
                interpolation='nearest', vmin=0, vmax=np.percentile(entropy_roi, 95))

cbar3 = plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
cbar3.set_label('Entropy [bits]', rotation=270, labelpad=15, color='white', fontsize=9)
cbar3.ax.tick_params(colors='white', labelsize=8)

ax3.set_title(f'Entropy at Boundary\n'
             f'Yellow/White = High Uncertainty\nRed/Black = Low Uncertainty',
             fontsize=11, fontweight='bold', color='white', pad=10)
ax3.axis('off')

# 统一样式
for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('black')
    for spine in ax.spines.values():
        spine.set_edgecolor('white')

# ========================================================================
# 第二行：统计图表
# ========================================================================

# 左图：Margin分布直方图
ax4 = fig.add_subplot(gs[1, 0])

all_margins = []
for band_mask in boundary_bands.values():
    band_flat = band_mask.flatten()
    margin_flat = margin.flatten()[band_flat]
    all_margins.extend(margin_flat)

all_margins = np.array(all_margins)

ax4.hist(all_margins, bins=50, color='dodgerblue', alpha=0.7, edgecolor='white')
ax4.axvline(np.median(all_margins), color='yellow', linestyle='--', linewidth=2,
           label=f'Median={np.median(all_margins):.3f}')
ax4.axvline(np.mean(all_margins), color='red', linestyle='--', linewidth=2,
           label=f'Mean={np.mean(all_margins):.3f}')

ax4.set_xlabel('Margin (p1-p2)', fontsize=10, fontweight='bold', color='white')
ax4.set_ylabel('Voxel Count', fontsize=10, fontweight='bold', color='white')
ax4.set_title('Margin Distribution at All Boundaries',
             fontsize=11, fontweight='bold', color='white', pad=10)
ax4.legend(fontsize=9, facecolor='black', edgecolor='white', labelcolor='white')
ax4.grid(True, alpha=0.3, color='white', linestyle=':')
ax4.tick_params(colors='white', labelsize=8)
ax4.set_facecolor('black')
for spine in ax4.spines.values():
    spine.set_edgecolor('white')

# 中图：Entropy分布直方图
ax5 = fig.add_subplot(gs[1, 1])

all_entropies = []
for band_mask in boundary_bands.values():
    band_flat = band_mask.flatten()
    entropy_flat = entropy_map.flatten()[band_flat]
    all_entropies.extend(entropy_flat)

all_entropies = np.array(all_entropies)

ax5.hist(all_entropies, bins=50, color='orangered', alpha=0.7, edgecolor='white')
ax5.axvline(np.percentile(all_entropies, 95), color='yellow', linestyle='--', linewidth=2,
           label=f'P95={np.percentile(all_entropies, 95):.3f}b')
ax5.axvline(np.median(all_entropies), color='cyan', linestyle='--', linewidth=2,
           label=f'Median={np.median(all_entropies):.3f}b')

ax5.set_xlabel('Entropy [bits]', fontsize=10, fontweight='bold', color='white')
ax5.set_ylabel('Voxel Count', fontsize=10, fontweight='bold', color='white')
ax5.set_title('Entropy Distribution at All Boundaries',
             fontsize=11, fontweight='bold', color='white', pad=10)
ax5.legend(fontsize=9, facecolor='black', edgecolor='white', labelcolor='white')
ax5.grid(True, alpha=0.3, color='white', linestyle=':')
ax5.tick_params(colors='white', labelsize=8)
ax5.set_facecolor('black')
for spine in ax5.spines.values():
    spine.set_edgecolor('white')

# 右图：Top-10 ROI的Soft Dice条形图
ax6 = fig.add_subplot(gs[1, 2])

top10_rois = df_roi_stats.head(10)
y_pos = np.arange(len(top10_rois))
soft_dices = top10_rois['soft_dice'].values
roi_labels = [name[:20] for name in top10_rois['roi_name'].values]

bars = ax6.barh(y_pos, soft_dices * 100, color='limegreen', alpha=0.8, edgecolor='white')

# 渐变颜色
for i, bar in enumerate(bars):
    bar.set_color(plt.cm.RdYlGn(soft_dices[i]))

# 标注数值
for i, (y, val) in enumerate(zip(y_pos, soft_dices)):
    ax6.text(val * 100 + 1, y, f'{val*100:.1f}%', 
            va='center', ha='left', fontsize=8, color='white', weight='bold')

ax6.set_yticks(y_pos)
ax6.set_yticklabels(roi_labels, fontsize=8)
ax6.set_xlabel('Soft Dice (%)', fontsize=10, fontweight='bold', color='white')
ax6.set_title('Soft Dice at Boundaries\n(Top-10 ROIs, tolerance=1 voxel)',
             fontsize=11, fontweight='bold', color='white', pad=10)
ax6.invert_yaxis()
ax6.set_xlim([0, max(soft_dices) * 110])
ax6.grid(axis='x', alpha=0.3, color='white', linestyle=':')
ax6.tick_params(colors='white', labelsize=8)
ax6.set_facecolor('black')
for spine in ax6.spines.values():
    spine.set_edgecolor('white')

# ========================================================================
# 总标题
# ========================================================================

fig.suptitle(f'{SUBJECT_ID} - Partial Volume (PV) Index at ROI Boundaries\n'
            f'{SPLIT_TYPE.upper()} Set - Boundary Band Analysis (±1 voxel)',
            fontsize=13, fontweight='bold', color='white', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# 保存
save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_pv_boundary_analysis.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
print(f"\n✓ 已保存PV边界分析图: {save_path.name}")

plt.show()
plt.close()

# ============================================================================
# 导出统计数据到CSV
# ============================================================================

print("\n导出边界条带统计到CSV...")

csv_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_pv_boundary_stats.csv'
df_roi_stats.to_csv(csv_path, index=False)
print(f"  ✓ 已保存边界统计: {csv_path.name}")

# ============================================================================
# 打印全局统计
# ============================================================================

print("\n全局边界条带统计:")
print(f"  总边界体素数: {sum(df_roi_stats['n_voxels']):,}")
print(f"  边界占ROI总体素比例: {sum(df_roi_stats['n_voxels']) / (region_mask > 0).sum() * 100:.2f}%")
print(f"\n  全局Margin统计:")
print(f"    Median: {np.median(all_margins):.4f}")
print(f"    Mean: {np.mean(all_margins):.4f}")
print(f"    P25: {np.percentile(all_margins, 25):.4f}")
print(f"    P75: {np.percentile(all_margins, 75):.4f}")
print(f"\n  全局Entropy统计:")
print(f"    Median: {np.median(all_entropies):.4f} bits")
print(f"    P95: {np.percentile(all_entropies, 95):.4f} bits")
print(f"    Mean: {np.mean(all_entropies):.4f} bits")
print(f"\n  全局Soft Dice统计:")
print(f"    Median: {df_roi_stats['soft_dice'].median():.4f}")
print(f"    Mean: {df_roi_stats['soft_dice'].mean():.4f}")
print(f"    Min: {df_roi_stats['soft_dice'].min():.4f}")
print(f"    Max: {df_roi_stats['soft_dice'].max():.4f}")

print(f"\n✓ PV边界条带评估完成！")

In [ ]:
print("绘制改进的可靠性图（硬标签 vs 软标签 ECE 对比）...\n")

from scipy.optimize import minimize

# ============================================================================
# 软标签版本的温度缩放和ECE计算（理论严谨版）
# ============================================================================

def temperature_scale_soft(probs, T):
    """
    软标签温度缩放：使用 p^(1/T) 重归一化
    
    这比 log(p)/T 更合理，因为：
    1. 没有原始logits时，p^(1/T) 是更自然的操作
    2. 保持概率的非负性和归一性
    3. T>1时软化分布，T<1时锐化分布
    """
    scaled = probs ** (1 / T)
    return scaled / (scaled.sum(axis=-1, keepdims=True) + 1e-12)

def compute_hard_ece(confidences, predictions, labels, n_bins=15):
    """
    硬标签ECE（传统方法，与Guo et al. 2017一致）
    
    准确度 = 预测正确的比例（0或1）
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0.0
    bin_data = {
        'bin_centers': [],
        'accuracies': [],
        'confidences': [],
        'counts': []
    }
    
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.sum() / len(confidences)
        
        if prop_in_bin > 0:
            # 硬标签：准确度 = 预测正确的比例
            accuracy_in_bin = (predictions[in_bin] == labels[in_bin]).mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            
            bin_data['bin_centers'].append((bin_lower + bin_upper) / 2)
            bin_data['accuracies'].append(accuracy_in_bin)
            bin_data['confidences'].append(avg_confidence_in_bin)
            bin_data['counts'].append(in_bin.sum())
    
    return ece, bin_data

def compute_soft_ece(confidences, predictions, gt_proba, n_bins=15):
    """
    软标签ECE（与软标签训练一致）
    
    准确度 = 预测类上的GT概率（而非0/1硬标签）
    这与软标签训练的目标一致
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0.0
    bin_data = {
        'bin_centers': [],
        'accuracies': [],
        'confidences': [],
        'counts': []
    }
    
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.sum() / len(confidences)
        
        if prop_in_bin > 0:
            # 软标签：准确度 = 预测类上的GT概率
            # 对于每个体素，取gt_proba[预测类别]作为"准确度"
            indices = np.arange(len(gt_proba))[in_bin]
            pred_classes = predictions[in_bin]
            gt_prob_on_pred = gt_proba[indices, pred_classes]
            
            accuracy_in_bin = gt_prob_on_pred.mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            
            bin_data['bin_centers'].append((bin_lower + bin_upper) / 2)
            bin_data['accuracies'].append(accuracy_in_bin)
            bin_data['confidences'].append(avg_confidence_in_bin)
            bin_data['counts'].append(in_bin.sum())
    
    return ece, bin_data

def soft_nll_loss(T, probs, gt_proba):
    """
    软标签NLL损失（用于优化温度参数）
    
    NLL = -Σ_k gt_proba[k] * log(pred[k])
    """
    T = T[0] if isinstance(T, np.ndarray) else T
    
    # 温度缩放：p^(1/T)
    scaled = temperature_scale_soft(probs, T)
    
    # 软标签NLL
    nll = -np.mean(np.sum(gt_proba * np.log(scaled + 1e-12), axis=-1))
    
    return nll

print("数据准备...")
print(f"  ROI体素数: {len(confidences):,}")

# ============================================================================
# 方法1：硬标签ECE（传统方法，与文献一致）
# ============================================================================

print("\n【方法1】硬标签ECE（传统方法）...")

# 温度缩放前（硬标签）
ece_hard_before, bins_hard_before = compute_hard_ece(
    confidences, predictions, gt_labels_flat, n_bins=15
)

# 温度优化（使用软标签NLL）
print("  优化温度参数（软标签NLL）...")
result_soft = minimize(
    soft_nll_loss,
    x0=[1.0],
    args=(pred_softmax_flat, gt_proba_flat),
    method='L-BFGS-B',
    bounds=[(0.05, 5.0)],
    options={'maxiter': 100}
)

T_soft_optimal = result_soft.x[0]

# 温度缩放后：p^(1/T)
probs_soft_scaled = temperature_scale_soft(pred_softmax_flat, T_soft_optimal)
confidences_soft_scaled = np.max(probs_soft_scaled, axis=1)
predictions_soft_scaled = np.argmax(probs_soft_scaled, axis=1)

# 温度缩放后（硬标签）
ece_hard_after, bins_hard_after = compute_hard_ece(
    confidences_soft_scaled, predictions_soft_scaled, gt_labels_flat, n_bins=15
)

print(f"  硬标签ECE: {ece_hard_before:.4f} → {ece_hard_after:.4f}")
print(f"  温度参数: T = {T_soft_optimal:.4f}")

# ============================================================================
# 方法2：软标签ECE（与训练一致）
# ============================================================================

print("\n【方法2】软标签ECE（与训练一致）...")

# 温度缩放前（软标签）
ece_soft_before, bins_soft_before = compute_soft_ece(
    confidences, predictions, gt_proba_flat, n_bins=15
)

# 温度缩放后（软标签）
ece_soft_after, bins_soft_after = compute_soft_ece(
    confidences_soft_scaled, predictions_soft_scaled, gt_proba_flat, n_bins=15
)

print(f"  软标签ECE: {ece_soft_before:.4f} → {ece_soft_after:.4f}")

# ============================================================================
# 可视化：硬标签 vs 软标签对比
# ============================================================================

print("\n绘制对比图...")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.patch.set_facecolor('black')

# ========================================================================
# 左图：硬标签ECE
# ========================================================================

ax1 = axes[0]

# 完美校准线
ax1.plot([0, 1], [0, 1], 'w--', linewidth=2, alpha=0.5, label='Perfect Calibration')

# 温度缩放前后（硬标签）
ax1.plot(bins_hard_before['confidences'], bins_hard_before['accuracies'],
        'o-', color='red', linewidth=2.5, markersize=8,
        label=f'Before T-Scaling (ECE={ece_hard_before:.4f})', alpha=0.85)

ax1.plot(bins_hard_after['confidences'], bins_hard_after['accuracies'],
        's-', color='cyan', linewidth=2.5, markersize=8,
        label=f'After T-Scaling (ECE={ece_hard_after:.4f}, T={T_soft_optimal:.3f})', alpha=0.85)

# 样本密度
ax2_1 = ax1.twinx()
bin_counts = np.array(bins_hard_before['counts'])
bin_counts_norm = bin_counts / bin_counts.max()
ax2_1.bar(bins_hard_before['bin_centers'], bin_counts_norm,
         width=1/15, alpha=0.15, color='gray')
ax2_1.set_ylabel('Normalized Density', fontsize=10, color='gray')
ax2_1.tick_params(axis='y', colors='gray', labelsize=8)
ax2_1.set_ylim([0, 1.2])

# 设置
ax1.set_xlabel('Confidence', fontsize=11, fontweight='bold', color='white')
ax1.set_ylabel('Accuracy (Hard Label)', fontsize=11, fontweight='bold', color='white')
ax1.set_title(f'Hard-Label ECE (Traditional)\n'
             f'Accuracy = Proportion Correct (0 or 1)\n'
             f'✓ Comparable with Literature (Guo et al. 2017)',
             fontsize=11, fontweight='bold', pad=10, color='white')

ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3, color='white', linestyle=':', linewidth=0.8)
ax1.tick_params(colors='white', labelsize=9)
ax1.set_facecolor('black')
ax1.legend(loc='upper left', fontsize=9, framealpha=0.9,
          edgecolor='white', facecolor='black', labelcolor='white')

for spine in ax1.spines.values():
    spine.set_edgecolor('white')
for spine in ax2_1.spines.values():
    spine.set_edgecolor('gray')

# 信息框
info_text_hard = (
    f"Hard-Label ECE:\n"
    f"  Before: {ece_hard_before:.4f}\n"
    f"  After:  {ece_hard_after:.4f}\n"
    f"  Reduction: {100*(ece_hard_before-ece_hard_after)/ece_hard_before:.1f}%\n"
    f"  T = {T_soft_optimal:.3f} (p^(1/T) scaling)"
)
ax1.text(0.98, 0.02, info_text_hard,
        transform=ax1.transAxes, fontsize=8,
        verticalalignment='bottom', horizontalalignment='right',
        color='yellow', weight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='black',
                 alpha=0.85, edgecolor='yellow', linewidth=1.5))

# ========================================================================
# 右图：软标签ECE
# ========================================================================

ax2 = axes[1]

# 完美校准线
ax2.plot([0, 1], [0, 1], 'w--', linewidth=2, alpha=0.5, label='Perfect Calibration')

# 温度缩放前后（软标签）
ax2.plot(bins_soft_before['confidences'], bins_soft_before['accuracies'],
        'o-', color='orangered', linewidth=2.5, markersize=8,
        label=f'Before T-Scaling (ECE={ece_soft_before:.4f})', alpha=0.85)

ax2.plot(bins_soft_after['confidences'], bins_soft_after['accuracies'],
        's-', color='limegreen', linewidth=2.5, markersize=8,
        label=f'After T-Scaling (ECE={ece_soft_after:.4f}, T={T_soft_optimal:.3f})', alpha=0.85)

# 样本密度
ax2_2 = ax2.twinx()
ax2_2.bar(bins_soft_before['bin_centers'], bin_counts_norm,
         width=1/15, alpha=0.15, color='gray')
ax2_2.set_ylabel('Normalized Density', fontsize=10, color='gray')
ax2_2.tick_params(axis='y', colors='gray', labelsize=8)
ax2_2.set_ylim([0, 1.2])

# 设置
ax2.set_xlabel('Confidence', fontsize=11, fontweight='bold', color='white')
ax2.set_ylabel('Accuracy (Soft Label)', fontsize=11, fontweight='bold', color='white')
ax2.set_title(f'Soft-Label ECE (Consistent with Training)\n'
             f'Accuracy = GT Probability on Predicted Class\n'
             f'✓ Aligned with Soft-Label Training Objective',
             fontsize=11, fontweight='bold', pad=10, color='white')

ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3, color='white', linestyle=':', linewidth=0.8)
ax2.tick_params(colors='white', labelsize=9)
ax2.set_facecolor('black')
ax2.legend(loc='upper left', fontsize=9, framealpha=0.9,
          edgecolor='white', facecolor='black', labelcolor='white')

for spine in ax2.spines.values():
    spine.set_edgecolor('white')
for spine in ax2_2.spines.values():
    spine.set_edgecolor('gray')

# 信息框
info_text_soft = (
    f"Soft-Label ECE:\n"
    f"  Before: {ece_soft_before:.4f}\n"
    f"  After:  {ece_soft_after:.4f}\n"
    f"  Reduction: {100*(ece_soft_before-ece_soft_after)/ece_soft_before:.1f}%\n"
    f"  T = {T_soft_optimal:.3f} (same as left)"
)
ax2.text(0.98, 0.02, info_text_soft,
        transform=ax2.transAxes, fontsize=8,
        verticalalignment='bottom', horizontalalignment='right',
        color='yellow', weight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='black',
                 alpha=0.85, edgecolor='yellow', linewidth=1.5))

# ========================================================================
# 总标题
# ========================================================================

fig.suptitle(f'{SUBJECT_ID} - Reliability Diagrams: Hard-Label vs Soft-Label ECE\n'
            f'{SPLIT_TYPE.upper()} Set - Temperature Scaling with p^(1/T) (Theoretically Rigorous)',
            fontsize=12, fontweight='bold', color='white', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# 保存
save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_reliability_hard_vs_soft.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='black')
print(f"\n✓ 已保存对比图: {save_path.name}")

plt.show()
plt.close()

# ============================================================================
# 打印对比统计
# ============================================================================

print("\n" + "="*70)
print("ECE对比总结:")
print("="*70)

print(f"\n温度参数: T = {T_soft_optimal:.4f}")
print(f"  (使用 p^(1/T) 缩放，而非 log(p)/T)")

print(f"\n硬标签ECE（与文献一致，Guo et al. 2017）:")
print(f"  Before: {ece_hard_before:.4f}")
print(f"  After:  {ece_hard_after:.4f}")
print(f"  Δ ECE:  {ece_hard_before - ece_hard_after:.4f} ({100*(ece_hard_before-ece_hard_after)/ece_hard_before:.1f}% reduction)")

print(f"\n软标签ECE（与训练一致，考虑GT不确定性）:")
print(f"  Before: {ece_soft_before:.4f}")
print(f"  After:  {ece_soft_after:.4f}")
print(f"  Δ ECE:  {ece_soft_before - ece_soft_after:.4f} ({100*(ece_soft_before-ece_soft_after)/ece_soft_before:.1f}% reduction)")

print(f"\n差异分析:")
print(f"  硬标签 vs 软标签 (Before): {ece_hard_before - ece_soft_before:.4f}")
print(f"  硬标签 vs 软标签 (After):  {ece_hard_after - ece_soft_after:.4f}")

if ece_soft_before < ece_hard_before:
    print(f"  → 软标签ECE更低，说明GT软标签本身有不确定性")
    print(f"  → 边界处的软混合被合理地视为'部分正确'")
else:
    print(f"  → 硬标签ECE更低（罕见情况）")

print("\n推荐:")
print("  - 论文中与文献对比：使用硬标签ECE")
print("  - 内部评估（与训练一致）：使用软标签ECE")
print("  - 两者都报告，说明区别，展示完整性")

print("\n✓ 可靠性图对比完成！")

---

## Figure 3: Top-K Prediction Analysis

**Figure 3a**: Top 1 predicted label overlay (colored by predicted class)

**Figure 3b**: Top 3 union coverage (voxel colored if ground truth label is in top 3; color=highest ranked label, opacity=max probability)

**Figure 3c**: Voxel-wise entropy heatmap (boundary zones show higher entropy, consistent with partial volume effect)


In [ ]:
print("绘制 Figure 3: Top-K Prediction Analysis...")# ============================================================================# Helper functions for Top-K analysis# ============================================================================def compute_top_k_coverage(pred_probs_3d, true_probs_3d, k=3):    """    Check if true label is in top-k predictions        Args:        pred_probs_3d: (Z, X, Y, 102) prediction probabilities        true_probs_3d: (Z, X, Y, 102) ground truth soft labels        k: top k        Returns:        in_top_k: (Z, X, Y) boolean array        top_k_labels: (Z, X, Y, k) top k predicted labels        top_k_probs: (Z, X, Y, k) top k predicted probabilities    """    Z, X, Y, n_classes = pred_probs_3d.shape        # Get top k indices and probabilities (descending order)    top_k_indices = np.argsort(pred_probs_3d, axis=-1)[..., -k:][:, :, :, ::-1]  # (Z, X, Y, k)    top_k_probs = np.take_along_axis(pred_probs_3d, top_k_indices, axis=-1)  # (Z, X, Y, k)        # Get true hard labels (argmax of soft labels)    true_labels = true_probs_3d.argmax(axis=-1)  # (Z, X, Y)        # Check if true label is in top k    true_labels_expanded = true_labels[..., np.newaxis]  # (Z, X, Y, 1)    in_top_k = np.any(top_k_indices == true_labels_expanded, axis=-1)  # (Z, X, Y)        return in_top_k, top_k_indices, top_k_probs# ============================================================================# Compute Top-K metrics (reuse existing variables where possible)# ============================================================================print("Computing top-k metrics...")# Reuse already computed variables from Cell 4:# - top1_class: Top 1 predicted class (Z, X, Y)# - p1: Top 1 probability (Z, X, Y)# - entropy_map: Entropy in bits (Z, X, Y)# Compute Top 3 coveragein_top3, top3_labels, top3_probs = compute_top_k_coverage(pred_softmax, gt_proba, k=3)# Compute global statisticsroi_mask_flat = region_mask > 0top3_coverage_global = in_top3[roi_mask_flat].mean()mean_entropy_global = entropy_map[roi_mask_flat].mean()print(f"  Global Top-3 Coverage: {top3_coverage_global:.1%}")print(f"  Mean Entropy (ROI): {mean_entropy_global:.3f} bits")# ============================================================================# Plot Figure 3 for each slice# ============================================================================for slice_idx in MANUAL_SLICES:    print(f"Processing slice z={slice_idx}...")        # Create figure with 3 subplots    fig, axes = plt.subplots(1, 3, figsize=(21, 7))    fig.suptitle(f"{SUBJECT_ID} - Axial Slice z={slice_idx} - Top-K Prediction Analysis",                 fontsize=16, fontweight="bold", y=0.98)        # Extract slice data    mask_slice = region_mask[slice_idx] > 0    pred_top1_slice = top1_class[slice_idx].copy().astype(float)  # Use existing top1_class    pred_max_prob_slice = p1[slice_idx].copy()  # Use existing p1    in_top3_slice = in_top3[slice_idx].copy()    top3_labels_slice = top3_labels[slice_idx]  # (X, Y, 3)    top3_probs_slice = top3_probs[slice_idx]  # (X, Y, 3)    entropy_slice = entropy_map[slice_idx].copy()  # Use existing entropy_map        # Set background to NaN for better visualization    pred_top1_slice[~mask_slice] = np.nan    entropy_slice[~mask_slice] = np.nan        # ========================================================================    # Figure 3a: Top 1 Predicted Label Overlay    # ========================================================================    ax = axes[0]        # Display predicted labels (only in ROI)    im1 = ax.imshow(pred_top1_slice, cmap=cmap_freesurfer, vmin=0, vmax=N_CLASSES-1,                    interpolation="nearest")        ax.set_title("(a) Top 1 Predicted Label", fontsize=14, fontweight="bold", pad=10)    ax.axis("off")        # Add statistics    roi_voxels = mask_slice.sum()    mean_conf = pred_max_prob_slice[mask_slice].mean()    ax.text(0.02, 0.98, f"ROI voxels: {roi_voxels}Mean confidence: {mean_conf:.3f}",           transform=ax.transAxes, fontsize=11, verticalalignment="top",           bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="gray"))        # ========================================================================    # Figure 3b: Top 3 Union Coverage    # ========================================================================    ax = axes[1]        # Create RGBA image    # Color: top 1 predicted label (from FreeSurfer colormap)    # Opacity: max probability if in top 3, red with high opacity if not    X_dim, Y_dim = pred_top1_slice.shape    coverage_rgba = np.zeros((X_dim, Y_dim, 4))        for i in range(X_dim):        for j in range(Y_dim):            if mask_slice[i, j]:                if in_top3_slice[i, j]:                    # Ground truth IS in top 3 - color by top 1 label                    label = int(top3_labels_slice[i, j, 0])  # Top 1 label                    if label < len(colors_list):                        color = colors_list[label]  # RGB from FreeSurfer                    else:                        color = (0.5, 0.5, 0.5)  # Gray for unknown                    # Opacity proportional to max probability                    alpha = float(top3_probs_slice[i, j, 0])  # Max probability                    coverage_rgba[i, j] = [*color, alpha]                else:                    # Ground truth NOT in top 3 - show in RED (error!)                    coverage_rgba[i, j] = [1.0, 0.0, 0.0, 0.9]        im2 = ax.imshow(coverage_rgba, interpolation="nearest")        ax.set_title("(b) Top 3 Union Coverage(Color=Top1 label, Opacity=Max prob, Red=Error)",                fontsize=14, fontweight="bold", pad=10)    ax.axis("off")        # Add statistics    coverage_rate = in_top3_slice[mask_slice].mean()    error_voxels = (~in_top3_slice[mask_slice]).sum()    ax.text(0.02, 0.98,            f"Top-3 Coverage: {coverage_rate:.1%}Errors: {error_voxels}/{roi_voxels}",           transform=ax.transAxes, fontsize=11, verticalalignment="top",           bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="gray"))        # ========================================================================    # Figure 3c: Voxel-wise Entropy Heatmap    # ========================================================================    ax = axes[2]        # Display entropy (only in ROI), entropy is in BITS    im3 = ax.imshow(entropy_slice, cmap="hot", interpolation="nearest",                    vmin=0, vmax=np.log2(N_CLASSES))  # Max entropy = log2(102) ≈ 6.67 bits        ax.set_title("(c) Voxel-wise Entropy Heatmap(Higher entropy at boundaries/PV zones)",                 fontsize=14, fontweight="bold", pad=10)    ax.axis("off")        # Add colorbar    cbar3 = plt.colorbar(im3, ax=ax, fraction=0.046, pad=0.04)    cbar3.set_label("Entropy (bits)", fontsize=11)        # Add statistics    entropy_roi = entropy_slice[mask_slice]    mean_ent = entropy_roi.mean()    std_ent = entropy_roi.std()    median_ent = np.median(entropy_roi)    max_possible = np.log2(N_CLASSES)    ax.text(0.02, 0.98,            f"Mean: {mean_ent:.3f} bitsMedian: {median_ent:.3f} bitsStd: {std_ent:.3f} bitsMax possible: {max_possible:.2f} bits",           transform=ax.transAxes, fontsize=10, verticalalignment="top",           bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="gray"))        # ========================================================================    # Save figure    # ========================================================================    plt.tight_layout()        save_path = OUTPUT_DIR / f"figure3_topk_analysis_z{slice_idx:03d}.png"    plt.savefig(save_path, dpi=300, bbox_inches="tight")    print(f"  ✓ Saved: {save_path.name}")        plt.show()    plt.close()print("✓ Figure 3 generation complete!")# ============================================================================# Generate global statistics summary# ============================================================================print("" + "="*80)print("Global Top-K Statistics Summary")print("="*80)# Compute top-k coverage for k=1,2,3,5,10for k in [1, 2, 3, 5, 10]:    in_topk, _, _ = compute_top_k_coverage(pred_softmax, gt_proba, k=k)    coverage = in_topk[roi_mask_flat].mean()    print(f"Top-{k:2d} Coverage: {coverage:.2%}")print(f"Entropy Statistics (ROI, in bits):")print(f"  Mean:   {entropy_map[roi_mask_flat].mean():.4f}")print(f"  Median: {np.median(entropy_map[roi_mask_flat]):.4f}")print(f"  Std:    {entropy_map[roi_mask_flat].std():.4f}")print(f"  Max:    {entropy_map[roi_mask_flat].max():.4f}")print(f"  Max possible: {np.log2(N_CLASSES):.2f} bits (uniform distribution)")print("="*80)

---

## LOSO Cross-Validation Results Summary

Aggregate statistics across all folds (Mean ± Std):
- Top-1, Top-3, Top-5 Accuracy
- NLL (Negative Log-Likelihood)
- ECE (Expected Calibration Error)
- Macro F1
- 3D Soft Dice (Macro)


In [ ]:
print("="*80)
print("LOSO Cross-Validation Results Summary")
print("="*80)

import json
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ============================================================================
# Configuration - Use BASE_RESULT_DIR from Cell 2
# ============================================================================

# BASE_RESULT_DIR should already be defined in Cell 2
# If running standalone, uncomment and set:
# BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/.../runs/loso_37fold")

print(f"Scanning directory: {BASE_RESULT_DIR}")
print(f"Directory exists: {BASE_RESULT_DIR.exists()}")

# ============================================================================
# Scan all fold directories
# ============================================================================

fold_dirs = sorted(BASE_RESULT_DIR.glob("fold_*"))
print(f"Found {len(fold_dirs)} fold directories")

if len(fold_dirs) == 0:
    print("⚠️  No fold directories found!")
    print("Please check BASE_RESULT_DIR path.")
else:
    # ========================================================================
    # Collect metrics from each fold
    # ========================================================================
    
    metrics_collection = {
        'fold_name': [],
        'test_subject': [],
        'top1_accuracy': [],
        'top3_accuracy': [],
        'top5_accuracy': [],
        'nll': [],
        'ece': [],
        'macro_f1': [],
        '3d_soft_dice_macro': []
    }
    
    print("Collecting metrics from each fold...")
    
    for fold_dir in fold_dirs:
        fold_name = fold_dir.name
        
        # Try to read run_summary.json first (preferred)
        summary_path = fold_dir / 'run_summary.json'
        test_metrics_path = fold_dir / 'metrics_test.json'
        
        metrics_data = None
        
        if summary_path.exists():
            try:
                with open(summary_path, 'r') as f:
                    data = json.load(f)
                
                # Extract from run_summary.json
                test_metrics = data.get('test_metrics', {})
                
                # Extract test subject from fold_name (format: fold_XX_test_subjectXXX)
                parts = fold_name.split('_')
                test_subject = '_'.join(parts[3:]) if len(parts) > 3 else 'unknown'
                
                metrics_data = {
                    'fold_name': fold_name,
                    'test_subject': test_subject,
                    'top1_accuracy': test_metrics.get('gross_accuracy', np.nan),
                    'top3_accuracy': test_metrics.get('top3_accuracy', np.nan),
                    'top5_accuracy': test_metrics.get('top5_accuracy', np.nan),
                    'nll': test_metrics.get('nll', np.nan),
                    'ece': test_metrics.get('soft_ece', np.nan),
                    'macro_f1': test_metrics.get('macro_f1', np.nan),
                    '3d_soft_dice_macro': data.get('test_3d_soft_dice_macro', np.nan)
                }
                
                print(f"  ✓ {fold_name}: Top1={metrics_data['top1_accuracy']:.4f}")
                
            except Exception as e:
                print(f"  ⚠️  {fold_name}: Error reading run_summary.json - {str(e)}")
                continue
        
        elif test_metrics_path.exists():
            try:
                with open(test_metrics_path, 'r') as f:
                    test_metrics = json.load(f)
                
                parts = fold_name.split('_')
                test_subject = '_'.join(parts[3:]) if len(parts) > 3 else 'unknown'
                
                metrics_data = {
                    'fold_name': fold_name,
                    'test_subject': test_subject,
                    'top1_accuracy': test_metrics.get('gross_accuracy', np.nan),
                    'top3_accuracy': test_metrics.get('top3_accuracy', np.nan),
                    'top5_accuracy': test_metrics.get('top5_accuracy', np.nan),
                    'nll': test_metrics.get('nll', np.nan),
                    'ece': test_metrics.get('soft_ece', np.nan),
                    'macro_f1': test_metrics.get('macro_f1', np.nan),
                    '3d_soft_dice_macro': np.nan  # Not available in metrics_test.json
                }
                
                print(f"  ✓ {fold_name}: Top1={metrics_data['top1_accuracy']:.4f} (from metrics_test.json)")
                
            except Exception as e:
                print(f"  ⚠️  {fold_name}: Error reading metrics_test.json - {str(e)}")
                continue
        
        else:
            print(f"  ⚠️  {fold_name}: No metrics files found")
            continue
        
        # Add to collection
        if metrics_data is not None:
            for key in metrics_collection.keys():
                metrics_collection[key].append(metrics_data[key])
    
    # ========================================================================
    # Create DataFrame and compute statistics
    # ========================================================================
    
    df_metrics = pd.DataFrame(metrics_collection)
    
    print(f"Successfully collected metrics from {len(df_metrics)} folds")
    
    # ========================================================================
    # Compute Mean ± Std
    # ========================================================================
    
    print("="*80)
    print("LOSO Cross-Validation Results (Mean ± Std)")
    print("="*80)
    
    metrics_to_summarize = [
        ('top1_accuracy', 'Top-1 Accuracy', True),  # True = percentage
        ('top3_accuracy', 'Top-3 Accuracy', True),
        ('top5_accuracy', 'Top-5 Accuracy', True),
        ('nll', 'NLL', False),
        ('ece', 'ECE (Soft)', False),
        ('macro_f1', 'Macro F1', False),
        ('3d_soft_dice_macro', '3D Soft Dice (Macro)', False)
    ]
    
    summary_data = []
    
    for metric_key, metric_name, is_percentage in metrics_to_summarize:
        values = df_metrics[metric_key].values
        
        # Filter out NaN values
        values_clean = values[~np.isnan(values)]
        
        if len(values_clean) > 0:
            mean_val = np.mean(values_clean)
            std_val = np.std(values_clean, ddof=1)  # Sample std
            min_val = np.min(values_clean)
            max_val = np.max(values_clean)
            median_val = np.median(values_clean)
            n_folds = len(values_clean)
            
            if is_percentage:
                # Convert to percentage
                summary_data.append({
                    'Metric': metric_name,
                    'Mean ± Std': f"{mean_val*100:.2f}% ± {std_val*100:.2f}%",
                    'Median': f"{median_val*100:.2f}%",
                    'Range': f"[{min_val*100:.2f}%, {max_val*100:.2f}%]",
                    'N': n_folds
                })
                
                print(f"\n{metric_name}:")
                print(f"  Mean ± Std: {mean_val*100:.2f}% ± {std_val*100:.2f}%")
                print(f"  Median: {median_val*100:.2f}%")
                print(f"  Range: [{min_val*100:.2f}%, {max_val*100:.2f}%]")
                print(f"  N: {n_folds}")
            else:
                summary_data.append({
                    'Metric': metric_name,
                    'Mean ± Std': f"{mean_val:.4f} ± {std_val:.4f}",
                    'Median': f"{median_val:.4f}",
                    'Range': f"[{min_val:.4f}, {max_val:.4f}]",
                    'N': n_folds
                })
                
                print(f"\n{metric_name}:")
                print(f"  Mean ± Std: {mean_val:.4f} ± {std_val:.4f}")
                print(f"  Median: {median_val:.4f}")
                print(f"  Range: [{min_val:.4f}, {max_val:.4f}]")
                print(f"  N: {n_folds}")
        else:
            print(f"\n{metric_name}: No valid data")
    
    print("\n" + "="*80)
    
    # ========================================================================
    # Display summary table
    # ========================================================================
    
    df_summary = pd.DataFrame(summary_data)
    
    print("\n\nSummary Table:")
    print("="*80)
    display(df_summary)
    
    # ========================================================================
    # Save summary to CSV
    # ========================================================================
    
    summary_csv_path = BASE_RESULT_DIR / 'loso_summary_statistics.csv'
    df_summary.to_csv(summary_csv_path, index=False)
    print(f"\n✓ Summary saved to: {summary_csv_path}")
    
    # Also save detailed per-fold data
    detailed_csv_path = BASE_RESULT_DIR / 'loso_detailed_metrics.csv'
    df_metrics.to_csv(detailed_csv_path, index=False)
    print(f"✓ Detailed metrics saved to: {detailed_csv_path}")
    
    print("\n" + "="*80)
    print("✓ LOSO Summary Complete!")
    print("="*80)


In [ ]:
print("Visualizing LOSO results...")

import matplotlib.pyplot as plt
import numpy as np

# ============================================================================
# Create boxplot visualization
# ============================================================================

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f'LOSO Cross-Validation Results Distribution (N={len(df_metrics)} folds)', 
             fontsize=16, fontweight='bold')

axes = axes.flatten()

plot_configs = [
    ('top1_accuracy', 'Top-1 Accuracy', True, axes[0]),
    ('top3_accuracy', 'Top-3 Accuracy', True, axes[1]),
    ('top5_accuracy', 'Top-5 Accuracy', True, axes[2]),
    ('nll', 'NLL', False, axes[3]),
    ('ece', 'ECE (Soft)', False, axes[4]),
    ('macro_f1', 'Macro F1', False, axes[5]),
    ('3d_soft_dice_macro', '3D Soft Dice (Macro)', False, axes[6])
]

for metric_key, title, is_percentage, ax in plot_configs:
    values = df_metrics[metric_key].values
    values_clean = values[~np.isnan(values)]
    
    if len(values_clean) > 0:
        # Convert to percentage if needed
        if is_percentage:
            values_clean = values_clean * 100
        
        # Boxplot
        bp = ax.boxplot([values_clean], vert=True, patch_artist=True, widths=0.6)
        bp['boxes'][0].set_facecolor('lightblue')
        bp['boxes'][0].set_alpha(0.7)
        
        # Scatter individual points
        x_pos = np.ones(len(values_clean)) + np.random.normal(0, 0.04, len(values_clean))
        ax.scatter(x_pos, values_clean, alpha=0.5, s=40, color='darkblue', edgecolors='black', linewidth=0.5)
        
        # Statistics lines
        mean_val = np.mean(values_clean)
        median_val = np.median(values_clean)
        ax.axhline(mean_val, color='red', linestyle='--', linewidth=2, 
                  label=f'Mean: {mean_val:.2f}{"%%" if is_percentage else ""}', alpha=0.8)
        
        # Title and labels
        ax.set_title(title, fontsize=12, fontweight='bold')
        ylabel = 'Value (%)' if is_percentage else 'Value'
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_xticks([1])
        ax.set_xticklabels([f'n={len(values_clean)}'])
        ax.legend(fontsize=9, loc='best')
        ax.grid(True, alpha=0.3, axis='y')
    else:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center', 
               transform=ax.transAxes, fontsize=12)
        ax.set_title(title, fontsize=12, fontweight='bold')

# Hide the last subplot if not used
axes[7].axis('off')

plt.tight_layout()

# Save figure
save_path = BASE_RESULT_DIR / 'loso_results_distribution.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✓ Visualization saved to: {save_path}")

plt.show()

# ============================================================================
# Print compact summary for paper/presentation
# ============================================================================

print("" + "="*80)
print("Compact Summary (for paper/presentation):")
print("="*80)
print()

for metric_key, metric_name, is_percentage in metrics_to_summarize:
    values = df_metrics[metric_key].values
    values_clean = values[~np.isnan(values)]
    
    if len(values_clean) > 0:
        mean_val = np.mean(values_clean)
        std_val = np.std(values_clean, ddof=1)
        
        if is_percentage:
            print(f"{metric_name:25s}: {mean_val*100:6.2f}% ± {std_val*100:5.2f}%")
        else:
            print(f"{metric_name:25s}: {mean_val:7.4f} ± {std_val:6.4f}")

print("="*80)


---

## Figure 4: Advanced Analysis - Confusion, Long-tail, and Class Mass Error

**Figure 4a**: Soft confusion matrix aggregated across LOSO (row-normalized, ordered by anatomical adjacency)

**Figure 4b**: Class volume vs performance scatter plot with LOWESS smoothing (illustrates long-tail effect)

**Figure 4c**: Class mass L1 error distribution


In [ ]:
print("Generating Figure 4: Advanced Analysis...")

# =============================================================================
# Imports
# =============================================================================
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import savgol_filter
from collections import defaultdict

# 依赖的外部变量（在外部环境中应已定义）：
# N_CLASSES, fold_dirs, LABEL_NAMES, SUBJECT_ID, gt_proba, region_mask, OUTPUT_DIR

# =============================================================================
# 1) Load and aggregate soft confusion matrices across all folds
# =============================================================================
print("Step 1: Loading soft confusion matrices from all folds...")

soft_cm_list = []
n_classes = N_CLASSES  # Should be 102

for fold_dir in fold_dirs:
    soft_cm_path = fold_dir / 'soft_confusion_test.csv'
    if soft_cm_path.exists():
        try:
            soft_cm = np.loadtxt(soft_cm_path, delimiter=',')
            if soft_cm.shape == (n_classes, n_classes):
                soft_cm_list.append(soft_cm)
                print(f"  ✓ {fold_dir.name}")
            else:
                print(f"  ⚠️  {fold_dir.name}: Incorrect shape {soft_cm.shape}")
        except Exception as e:
            print(f"  ⚠️  {fold_dir.name}: Error - {str(e)}")
    else:
        print(f"  ⚠️  {fold_dir.name}: soft_confusion_test.csv not found")

if len(soft_cm_list) == 0:
    print("❌ No soft confusion matrices found! Skipping Figure 4a.")
    aggregated_soft_cm = None
else:
    # Aggregate (average) across all folds
    aggregated_soft_cm = np.mean(soft_cm_list, axis=0)
    print(f"✓ Successfully aggregated {len(soft_cm_list)} soft confusion matrices")
    print(f"  Matrix shape: {aggregated_soft_cm.shape}")
    print(f"  Diagonal mean (correct classification): {np.diag(aggregated_soft_cm).mean():.4f}")

# =============================================================================
# 2) Compute per-class performance and volume
# =============================================================================
print("Step 2: Computing per-class performance and volume...")

# Initialize storage
per_class_stats = {
    'class_id': list(range(n_classes)),
    'class_name': [],
    'recall': [],      # Top-1 recall per class (from confusion matrix)
    'precision': [],
    'f1': [],
    'volume': []       # Total voxel count (soft volume)
}

# Compute recall and precision from aggregated confusion matrix
if aggregated_soft_cm is not None:
    for class_id in range(n_classes):
        # Recall: diagonal element (correctly classified)
        recall = aggregated_soft_cm[class_id, class_id]

        # Precision: correct predictions / all predictions for this class
        col_sum = aggregated_soft_cm[:, class_id].sum()
        precision = aggregated_soft_cm[class_id, class_id] / col_sum if col_sum > 0 else 0

        # F1 score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        per_class_stats['recall'].append(recall)
        per_class_stats['precision'].append(precision)
        per_class_stats['f1'].append(f1)
else:
    # Fill with NaN if no confusion matrix
    per_class_stats['recall'] = [np.nan] * n_classes
    per_class_stats['precision'] = [np.nan] * n_classes
    per_class_stats['f1'] = [np.nan] * n_classes

# Compute class volumes from training data
# We'll use the current subject's data as a proxy (or you can aggregate across all subjects)
print("  Computing class volumes from ground truth data...")
class_volumes = np.zeros(n_classes)

# Use gt_proba which is already loaded from Cell 4
if 'gt_proba' in globals() and 'region_mask' in globals():
    # Compute volume for current subject
    roi_mask = region_mask > 0
    for class_id in range(n_classes):
        class_volumes[class_id] = gt_proba[..., class_id][roi_mask].sum()
    print(f"  ✓ Class volumes computed from current subject: {SUBJECT_ID}")
else:
    print("  ⚠️  gt_proba or region_mask not found, using dummy volumes")
    class_volumes = np.random.rand(n_classes) * 1000  # Dummy data

per_class_stats['volume'] = class_volumes.tolist()

# Add class names
for class_id in range(n_classes):
    class_name = LABEL_NAMES.get(class_id, f'Class_{class_id}')
    per_class_stats['class_name'].append(class_name)

print(f"  ✓ Volume range: [{class_volumes.min():.0f}, {class_volumes.max():.0f}]")
print(f"  ✓ Total volume: {class_volumes.sum():.0f}")

# Create DataFrame
df_per_class = pd.DataFrame(per_class_stats)

# =============================================================================
# 3) Collect class mass L1 errors across folds
# =============================================================================
print("Step 3: Collecting class mass L1 errors...")

class_mass_errors = defaultdict(list)

for fold_dir in fold_dirs:
    metrics_path = fold_dir / 'metrics_test.json'
    if metrics_path.exists():
        try:
            with open(metrics_path, 'r') as f:
                metrics = json.load(f)

            # Extract class_mass_top_errors
            top_errors = metrics.get('class_mass_top_errors', [])
            for error_info in top_errors:
                class_id = error_info['class_id']
                abs_error = error_info['abs_error']
                class_mass_errors[class_id].append(abs_error)
        except Exception:
            # Silently skip errors
            pass

# Compute average L1 error per class
avg_class_mass_errors = []
for class_id in range(n_classes):
    if class_id in class_mass_errors and len(class_mass_errors[class_id]) > 0:
        avg_error = np.mean(class_mass_errors[class_id])
    else:
        avg_error = 0.0
    avg_class_mass_errors.append(avg_error)

print(f"  ✓ Collected errors for {len(class_mass_errors)} classes")

# =============================================================================
# 4) Define anatomical ordering (simplified by volume)
# =============================================================================
print("Step 4: Ordering classes by volume (proxy for anatomical adjacency)...")

# For simplicity, we'll order by volume (largest to smallest)
# In a real scenario, you'd use anatomical hierarchy from FreeSurfer
top_n_classes = 50  # Show top 50 classes by volume
top_classes_by_volume = np.argsort(class_volumes)[::-1][:top_n_classes]
anatomical_order = top_classes_by_volume

print(f"  Using top {top_n_classes} classes by volume")

# =============================================================================
# 5) Plot Figure 4
# =============================================================================
print("Step 5: Plotting Figure 4...")

fig = plt.figure(figsize=(21, 7))
gs = fig.add_gridspec(1, 3, wspace=0.3)

# ------------------------------ Figure 4a ------------------------------------
ax1 = fig.add_subplot(gs[0, 0])

if aggregated_soft_cm is not None:
    # Extract subset for top classes
    cm_subset = aggregated_soft_cm[np.ix_(anatomical_order, anatomical_order)]

    # Plot
    im = ax1.imshow(cm_subset, cmap='hot', aspect='auto', vmin=0, vmax=1)
    ax1.set_title(
        f'(a) Aggregated Soft Confusion Matrix '
        f'(Top {top_n_classes} classes by volum)',
        fontsize=13, fontweight='bold', pad=10
    )
    ax1.set_xlabel('Predicted Class', fontsize=11)
    ax1.set_ylabel('True Class', fontsize=11)

    # Colorbar
    cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    cbar.set_label('Probability', fontsize=10)

    # Ticks (show every 10th)
    tick_positions = np.arange(0, len(anatomical_order), 10)
    ax1.set_xticks(tick_positions)
    ax1.set_yticks(tick_positions)
    ax1.set_xticklabels(anatomical_order[tick_positions], fontsize=7, rotation=90)
    ax1.set_yticklabels(anatomical_order[tick_positions], fontsize=7)

    # Add diagonal line
    ax1.plot(
        [0, len(anatomical_order) - 1],
        [0, len(anatomical_order) - 1],
        'b--', linewidth=1, alpha=0.3
    )
else:
    ax1.text(
        0.5, 0.5, 'No confusion matrix available',
        ha='center', va='center', transform=ax1.transAxes, fontsize=12
    )
    ax1.set_title('(a) Aggregated Soft Confusion Matrix', fontsize=13, fontweight='bold')

# ------------------------------ Figure 4b ------------------------------------
ax2 = fig.add_subplot(gs[0, 1])

# Filter out zero-volume classes
valid_mask = df_per_class['volume'] > 0
df_valid = df_per_class[valid_mask].copy()

if len(df_valid) > 0:
    x = df_valid['volume'].values
    y = df_valid['recall'].values  # Use Top-1 recall

    # Sort by volume for smooth curve
    sorted_indices = np.argsort(x)
    x_sorted = x[sorted_indices]
    y_sorted = y[sorted_indices]

    # Scatter plot (log x-axis)
    scatter = ax2.scatter(
        x, y, alpha=0.6, s=50, c=y, cmap='viridis',
        edgecolors='black', linewidth=0.5, zorder=3
    )

    # LOWESS smoothing (use Savitzky-Golay filter as approximation)
    if len(x_sorted) > 10:
        window = min(31, len(x_sorted) // 3)
        if window % 2 == 0:
            window += 1  # Must be odd
        if window >= 3:
            try:
                y_smooth = savgol_filter(y_sorted, window_length=window, polyorder=2)
                ax2.plot(
                    x_sorted, y_smooth, 'r-', linewidth=3,
                    label='LOWESS trend', alpha=0.8, zorder=4
                )
            except Exception:
                pass  # Skip if smoothing fails

        ax2.set_xscale('log')

    ax2.set_xlabel('Class Volume (log scale)', fontsize=11)
    ax2.set_ylabel('Top-1 Recall', fontsize=11)
    ax2.set_title(
        '(b) Long-tail Effect: Volume vs Performance',
        fontsize=13, fontweight='bold', pad=10
    )
    ax2.grid(True, alpha=0.3, which='both')
    ax2.legend(fontsize=9)

    # Colorbar
    cbar2 = plt.colorbar(scatter, ax=ax2, fraction=0.046, pad=0.04)
    cbar2.set_label('Recall', fontsize=10)

    # Add correlation
    correlation = np.corrcoef(np.log10(x), y)[0, 1]
    ax2.text(
        0.05, 0.95, f'Log-correlation: {correlation:.3f}',
        transform=ax2.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )
else:
    ax2.text(
        0.5, 0.5, 'No valid data', ha='center', va='center',
        transform=ax2.transAxes, fontsize=12
    )
    ax2.set_title('(b) Long-tail Effect', fontsize=13, fontweight='bold')

# ------------------------------ Figure 4c ------------------------------------
ax3 = fig.add_subplot(gs[0, 2])

# Filter out zero errors
errors_nonzero = [e for e in avg_class_mass_errors if e > 0]

if len(errors_nonzero) > 0:
    # Histogram
    ax3.hist(errors_nonzero, bins=30, alpha=0.7, color='salmon', edgecolor='black')
    ax3.axvline(
        np.mean(errors_nonzero), color='red', linestyle='--', linewidth=2,
        label=f'Mean: {np.mean(errors_nonzero):.1f}'
    )
    ax3.axvline(
        np.median(errors_nonzero), color='green', linestyle='--', linewidth=2,
        label=f'Median: {np.median(errors_nonzero):.1f}'
    )

    ax3.set_xlabel('L1 Error (Absolute)', fontsize=11)
    ax3.set_ylabel('Number of Classes', fontsize=11)
    ax3.set_title(
        '(c) Class Mass L1 Error Distribution',
        fontsize=13, fontweight='bold', pad=10
    )
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')

    # Statistics
    stats_text = (
        f'N classes: {len(errors_nonzero)}'
        f'\nMax: {np.max(errors_nonzero):.1f}'
    )
    ax3.text(
        0.98, 0.98, stats_text, transform=ax3.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )
else:
    # Show all errors (including zeros)
    ax3.hist(avg_class_mass_errors, bins=30, alpha=0.7, color='salmon', edgecolor='black')
    ax3.set_xlabel('L1 Error (Absolute)', fontsize=11)
    ax3.set_ylabel('Number of Classes', fontsize=11)
    ax3.set_title(
        '(c) Class Mass L1 Error Distribution',
        fontsize=13, fontweight='bold', pad=10
    )
    ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# Save figure
save_path = OUTPUT_DIR / 'figure4_advanced_analysis.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.show()

# =============================================================================
# Print summary statistics
# =============================================================================
print("" + "="*80)
print("Figure 4 Summary Statistics")
print("="*80)

if aggregated_soft_cm is not None:
    diag_mean = np.diag(aggregated_soft_cm).mean()
    off_diag_mean = (
        aggregated_soft_cm.sum() - np.diag(aggregated_soft_cm).sum()
    ) / (n_classes * (n_classes - 1))
    print(f"Confusion Matrix (aggregated):")
    print(f"  Diagonal mean (correct rate): {diag_mean:.4f}")
    print(f"  Off-diagonal mean (confusion rate): {off_diag_mean:.4f}")
    print(f"  Diagonal / Off-diagonal ratio: {diag_mean / off_diag_mean:.2f}x")

if len(df_valid) > 0:
    print("Long-tail Effect:")
    print(f"  Max volume class: {df_valid['volume'].max():.0f} voxels")
    print(f"  Min volume class: {df_valid['volume'].min():.0f} voxels")
    print(f"  Volume ratio: {df_valid['volume'].max() / df_valid['volume'].min():.1f}x")
    print(f"  Volume-performance log-correlation: {correlation:.3f}")

if len(errors_nonzero) > 0:
    print("Class Mass L1 Error:")
    print(f"  Mean L1 error: {np.mean(errors_nonzero):.2f}")
    print(f"  Median L1 error: {np.median(errors_nonzero):.2f}")
    print(f"  Max L1 error: {np.max(errors_nonzero):.2f}")
    print(f"  Classes with error: {len(errors_nonzero)}/{n_classes}")

print("="*80)
print("✓ Figure 4 generation complete!")


In [ ]:
# ============================================
# 合并版绘图（2×2 和 1×4）
# 依赖：已计算出以下变量
# - bin_data_before, bin_data_after, ece_before, ece_after, T_optimal, confidences
# - coverages, nll_at_coverage, brier_at_coverage, error_at_coverage
# - aurc_nll, aurc_brier, aurc_error
# - SUBJECT_ID, SPLIT_TYPE, OUTPUT_DIR
# ============================================

import matplotlib.pyplot as plt
import numpy as np

# ---------- 小工具函数：在指定轴上画“可靠性图” ----------
def draw_reliability(ax, bin_before, bin_after, ece_b, ece_a, T_opt, n_bins=15, voxels_cnt=None):
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=2, alpha=0.5, label='Perfect', color='white')
    ax.plot(bin_before['confidences'], bin_before['accuracies'],
            marker='o', linestyle='-', linewidth=2.0, markersize=6, alpha=0.9,
            label=f'Before (ECE={ece_b:.4f})', color='red')
    ax.plot(bin_after['confidences'], bin_after['accuracies'],
            marker='s', linestyle='-', linewidth=2.0, markersize=6, alpha=0.9,
            label=f'After  (ECE={ece_a:.4f}, T={T_opt:.3f})', color='cyan')

    # 样本密度（副轴）
    ax2 = ax.twinx()
    bin_counts = np.array(bin_before['counts'])
    if bin_counts.max() > 0:
        bin_counts_norm = bin_counts / bin_counts.max()
    else:
        bin_counts_norm = bin_counts
    ax2.bar(bin_before['bin_centers'], bin_counts_norm, width=1/n_bins, alpha=0.20, color='gray', label='Density')
    ax2.set_ylim(0, 1.2)
    ax2.set_ylabel('Normalized Density', fontsize=9, color='gray')
    ax2.tick_params(axis='y', colors='gray', labelsize=8)

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Confidence', fontsize=10, fontweight='bold', color='white')
    ax.set_ylabel('Accuracy', fontsize=10, fontweight='bold', color='white')
    ax.grid(True, alpha=0.3, color='white', linestyle=':')
    ax.tick_params(colors='white', labelsize=9)
    ax.set_facecolor('black')
    for sp in ax.spines.values(): sp.set_edgecolor('white')
    for sp in ax2.spines.values(): sp.set_edgecolor('gray')

    # 信息角标
    if voxels_cnt is None:
        voxels_cnt = sum(bin_before['counts'])
    info = (f"ECE: {ece_b:.4f} → {ece_a:.4f}\n"
            f"T = {T_opt:.3f}\n"
            f"Voxels: {voxels_cnt:,}")
    ax.text(0.98, 0.02, info, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8, color='yellow',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='black', alpha=0.85, edgecolor='yellow'))

    # 合并图里避免太多图例拥挤：放右上
    ax.legend(loc='upper left', fontsize=8, framealpha=0.9,
              edgecolor='white', facecolor='black', labelcolor='white')


# ---------- 小工具函数：在指定轴上画单条“风险-覆盖曲线” ----------
def draw_risk_curve(ax, coverages, risk, title, aurc):
    # 主曲线
    ax.plot(coverages * 100, risk, linewidth=2.0, alpha=0.9, label=f'AURC={aurc:.4f}')
    ax.fill_between(coverages * 100, risk, alpha=0.20)

    # 标注关键 coverage
    for cov_point in [0.5, 0.7, 0.9, 1.0]:
        if cov_point <= coverages.max():
            idx = np.argmin(np.abs(coverages - cov_point))
            rv = risk[idx]
            ax.plot(cov_point * 100, rv, 'o', color='yellow', markersize=6,
                    markeredgecolor='white', markeredgewidth=1.2, zorder=5)
            ax.annotate(f'{int(cov_point*100)}%: {rv:.3f}',
                        xy=(cov_point * 100, rv),
                        xytext=(cov_point * 100 + 2, rv + (risk.max() - risk.min()) * 0.06),
                        fontsize=7.5, color='yellow',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.75, edgecolor='yellow'),
                        arrowprops=dict(arrowstyle='->', color='yellow', lw=1))

    ax.set_xlim(0, 100)
    ax.set_ylim(0, risk.max() * 1.12 if risk.max() > 0 else 1.0)
    ax.set_xlabel('Coverage (% of voxels kept)', fontsize=10, fontweight='bold', color='white')
    ax.set_ylabel(f'Risk ({title})', fontsize=10, fontweight='bold', color='white')
    ax.set_title(title, fontsize=11, fontweight='bold', color='white', pad=8)
    ax.grid(True, alpha=0.3, color='white', linestyle=':')
    ax.tick_params(colors='white', labelsize=9)
    ax.set_facecolor('black')
    for sp in ax.spines.values(): sp.set_edgecolor('white')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9,
              edgecolor='white', facecolor='black', labelcolor='white')

    # 补充说明
    ax.text(0.02, 0.98, "Lower AURC is better", transform=ax.transAxes,
            ha='left', va='top', fontsize=8, color='cyan',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.8, edgecolor='cyan'))


# ========= 版本 A：2×2 合并图 =========
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('black')

# (0,0) 可靠性图
draw_reliability(axes[0, 0], bin_data_before, bin_data_after, ece_before, ece_after, T_optimal, n_bins=15, voxels_cnt=len(confidences))
axes[0, 0].set_title('Reliability Diagram (Before vs After T-Scaling)', fontsize=12, fontweight='bold', color='white', pad=10)

# (0,1) NLL
draw_risk_curve(axes[0, 1], coverages, nll_at_coverage, 'NLL (Neg. Log-Likelihood)', aurc_nll)

# (1,0) Brier
draw_risk_curve(axes[1, 0], coverages, brier_at_coverage, 'Brier Score', aurc_brier)

# (1,1) Error
draw_risk_curve(axes[1, 1], coverages, error_at_coverage, 'Error Rate', aurc_error)

fig.suptitle(f'{SUBJECT_ID} — Calibration & Selective Prediction ({SPLIT_TYPE.upper()} Set)',
             fontsize=14, fontweight='bold', color='white', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])

save_path_2x2 = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_combined_2x2.png'
plt.savefig(save_path_2x2, dpi=220, bbox_inches='tight', facecolor='black')
print(f"✓ 已保存合并图（2×2）: {save_path_2x2.name}")
plt.show()
plt.close(fig)


# ========= 版本 B：1×4 合并图 =========
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
fig.patch.set_facecolor('black')

# (0) 可靠性图
draw_reliability(axes[0], bin_data_before, bin_data_after, ece_before, ece_after, T_optimal, n_bins=15, voxels_cnt=len(confidences))
axes[0].set_title('Reliability (Before vs After)', fontsize=12, fontweight='bold', color='white', pad=10)

# (1) NLL
draw_risk_curve(axes[1], coverages, nll_at_coverage, 'NLL', aurc_nll)
# (2) Brier
draw_risk_curve(axes[2], coverages, brier_at_coverage, 'Brier', aurc_brier)
# (3) Error
draw_risk_curve(axes[3], coverages, error_at_coverage, 'Error', aurc_error)

fig.suptitle(f'{SUBJECT_ID} — Calibration & Risk–Coverage (1×4) [{SPLIT_TYPE.upper()}]',
             fontsize=14, fontweight='bold', color='white', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])

save_path_1x4 = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_combined_1x4.png'
plt.savefig(save_path_1x4, dpi=220, bbox_inches='tight', facecolor='black')
print(f"✓ 已保存合并图（1×4）: {save_path_1x4.name}")
plt.show()
plt.close(fig)


In [ ]:
# ====== A. 纯不确定性热图（确定性 Top-K 标注 + 白底 + 科学插值） ======
from math import hypot
import numpy as np
import matplotlib.pyplot as plt

def _topk_peaks(arr2d, mask2d, k=3, min_dist=10):
    """确定性 Top-K 最大值坐标，带最小间距约束（像素）"""
    vals = arr2d.copy()
    vals[~(mask2d.astype(bool))] = -np.inf
    coords = []
    for _ in range(k):
        idx = np.argmax(vals)
        if not np.isfinite(vals.ravel()[idx]):  # 没有更多候选
            break
        x, y = np.unravel_index(idx, vals.shape)
        if all(hypot(x-x0, y-y0) >= min_dist for x0, y0 in coords):
            coords.append((x, y))
        vals[max(0,x-min_dist):x+min_dist+1, max(0,y-min_dist):y+min_dist+1] = -np.inf
    return coords

print("绘制纯不确定性热图...\n")

for slice_idx in MANUAL_SLICES:
    print(f"处理切片 {slice_idx}...")
    anatomy_slice = anatomy_img[slice_idx]
    mask_slice = region_mask[slice_idx].astype(bool)
    entropy_slice = entropy_map_masked[slice_idx]
    margin_slice = margin[slice_idx]

    # 归一化背景（灰度）
    anatomy_norm = (anatomy_slice - anatomy_slice.min()) / (anatomy_slice.ptp() + 1e-8)

    # ROI 内统计（确定性 vmax）
    entropy_roi = entropy_slice[mask_slice]
    if entropy_roi.size == 0:
        print("  ⚠️ ROI 内无体素，跳过该切片"); continue
    vmax = np.quantile(entropy_roi, 0.95)  # 95% 分位数，抗极值

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))  # 白底
    # 背景：MPRAGE 灰度（保边界）
    ax.imshow(anatomy_norm, cmap='gray', origin='upper', interpolation='nearest', alpha=0.30)
    # 主图：熵伪彩（保边界）
    entropy_masked = np.ma.masked_where(~mask_slice, entropy_slice)
    im = ax.imshow(entropy_masked, cmap='hot', origin='upper', interpolation='nearest', vmin=0, vmax=vmax, alpha=0.85)

    # 色条
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Entropy H(p) [bits]', rotation=270, labelpad=14)

    # 置信度等值线 Δp = p1 - p2（绝对阈值，越大越“自信”）
    abs_levels = [0.2, 0.4, 0.6]
    lo, hi = np.nanmin(margin_slice), np.nanmax(margin_slice)
    levels = [lv for lv in abs_levels if (lv >= lo and lv <= hi)]
    if levels:
        cs = ax.contour(margin_slice, levels=levels, colors='white', linewidths=1.2, linestyles='solid', alpha=0.8)
        ax.clabel(cs, inline=True, fontsize=8, fmt=r'$\Delta p$=%.1f')
    # ROI 边界
    ax.contour(mask_slice.astype(float), levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--', alpha=0.8)

    # Top-K 最高不确定性（确定性选择 + 最小间距）
    for (x, y) in _topk_peaks(entropy_slice, mask_slice, k=3, min_dist=12):
        h_val = float(entropy_slice[x, y]); m_val = float(margin_slice[x, y])
        ax.plot(y, x, marker='*', markersize=10, markeredgewidth=1.2, markeredgecolor='black', color='yellow')
        ax.annotate(f'H={h_val:.2f} bits\nΔp={m_val:.2f}', xy=(y, x), xytext=(y+12, x-12),
                    fontsize=7.5, color='black',
                    bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.9),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.0))

    ax.set_title(f'{SUBJECT_ID}  |  {SLICE_AXIS.capitalize()} slice {slice_idx}\n'
                 'Hot = higher entropy; white contours = Δp (p1−p2)', fontsize=10)
    ax.axis('off')

    plt.tight_layout()
    save_path = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_uncertainty_slice{slice_idx:03d}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ 已保存: {save_path.name}")
    plt.close(fig)

print("\n✓ 不确定性热图完成！")


In [ ]:
# ====== B. 可靠性 + 风险-覆盖（白底 + Wilson CI + AURC 双曲线） ======
import numpy as np
import matplotlib.pyplot as plt

def _wilson_ci(k, n, z=1.95996398454):
    if n <= 0: return (np.nan, np.nan)
    phat = k / n
    denom = 1.0 + z**2 / n
    center = (phat + z**2/(2*n)) / denom
    margin = z * np.sqrt((phat*(1-phat)/n) + (z**2/(4*n**2))) / denom
    return center - margin, center + margin

def draw_reliability(ax, bin_before, bin_after, ece_b, ece_a, T_opt, n_bins=15):
    # 45° 线
    ax.plot([0,1],[0,1], ls='--', lw=1.2, color='0.6', label='Perfect')

    # before
    cb, ab, nb = np.array(bin_before['confidences']), np.array(bin_before['accuracies']), np.array(bin_before['counts'])
    ax.plot(cb, ab, marker='o', lw=1.2, label=f'Before (ECE={ece_b:.3f})')
    for c,a,n in zip(cb,ab,nb):
        lo, hi = _wilson_ci(int(round(a*n)), int(n))
        if np.isfinite(lo):
            ax.vlines(c, lo, hi, lw=0.8, color='C0', alpha=0.7)

    # after
    ca, aa, na = np.array(bin_after['confidences']), np.array(bin_after['accuracies']), np.array(bin_after['counts'])
    ax.plot(ca, aa, marker='s', lw=1.2, label=f'After (T={T_opt:.2f}; ECE={ece_a:.3f})')
    for c,a,n in zip(ca,aa,na):
        lo, hi = _wilson_ci(int(round(a*n)), int(n))
        if np.isfinite(lo):
            ax.vlines(c, lo, hi, lw=0.8, color='C1', alpha=0.7)

    # 样本密度（灰条）
    counts = nb
    if counts.sum() > 0:
        ax.bar(bin_before['bin_centers'], counts / counts.max(), width=1/n_bins, alpha=0.15, color='0.5', label='Density (norm.)')

    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.legend(frameon=False, loc='upper left')
    ax.set_title('Reliability (Before vs After)')

def _aurc(coverage, risk):
    coverage = np.asarray(coverage); risk = np.asarray(risk)
    # 保证 coverage 单调
    order = np.argsort(coverage)
    return float(np.trapz(risk[order], coverage[order]))

def draw_risk_curve(ax, cov_after, risk_after, title, cov_before=None, risk_before=None):
    """支持 before/after；若未提供 before，只画 after"""
    if cov_before is not None and risk_before is not None:
        aurc_b = _aurc(cov_before, risk_before)
        ax.plot(cov_before*100, risk_before, lw=1.5, label=f'Before (AURC={aurc_b:.3f})')
    aurc_a = _aurc(cov_after, risk_after)
    ax.plot(cov_after*100, risk_after, lw=1.8, label=f'After (AURC={aurc_a:.3f})')

    # 标注关键 coverage
    for cp in (0.5, 0.7, 0.9, 1.0):
        if cp <= np.max(cov_after):
            i = np.argmin(np.abs(cov_after - cp))
            ax.plot(cp*100, risk_after[i], 'o', ms=3)
            ax.annotate(f'{int(cp*100)}%', (cp*100, risk_after[i]),
                        textcoords="offset points", xytext=(2, 4), fontsize=7)

    ax.set_xlim(0,100)
    ax.set_xlabel('Coverage (% of voxels kept)'); ax.set_ylabel(title)
    ax.legend(frameon=False, loc='best')
    ax.set_title(title)

# -------- 合并图（2×2 与 1×4），如果有 *_before 数组就自动画双曲线 --------
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
draw_reliability(axes[0,0], bin_data_before, bin_data_after, ece_before, ece_after, T_optimal, n_bins=15)

# Risk–coverage（这里演示 NLL；若你已计算 *_before 数组，就传进去）
draw_risk_curve(axes[0,1], coverages, nll_at_coverage, 'NLL',
                cov_before=globals().get('coverages_before'),
                risk_before=globals().get('nll_at_coverage_before'))
draw_risk_curve(axes[1,0], coverages, brier_at_coverage, 'Brier score',
                cov_before=globals().get('coverages_before'),
                risk_before=globals().get('brier_at_coverage_before'))
draw_risk_curve(axes[1,1], coverages, error_at_coverage, 'Error rate',
                cov_before=globals().get('coverages_before'),
                risk_before=globals().get('error_at_coverage_before'))

fig.suptitle(f'{SUBJECT_ID} — Calibration & Selective Prediction ({SPLIT_TYPE.upper()} set)', y=0.98, fontsize=12)
plt.tight_layout(rect=[0,0,1,0.96])
save_path_2x2 = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_combined_2x2.png'
plt.savefig(save_path_2x2, dpi=300, bbox_inches='tight')
print(f"✓ 已保存合并图（2×2）: {save_path_2x2.name}")
plt.close(fig)

# 横向 1×4
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
draw_reliability(axes[0], bin_data_before, bin_data_after, ece_before, ece_after, T_optimal, n_bins=15)
draw_risk_curve(axes[1], coverages, nll_at_coverage, 'NLL',
                cov_before=globals().get('coverages_before'),
                risk_before=globals().get('nll_at_coverage_before'))
draw_risk_curve(axes[2], coverages, brier_at_coverage, 'Brier',
                cov_before=globals().get('coverages_before'),
                risk_before=globals().get('brier_at_coverage_before'))
draw_risk_curve(axes[3], coverages, error_at_coverage, 'Error',
                cov_before=globals().get('coverages_before'),
                risk_before=globals().get('error_at_coverage_before'))
fig.suptitle(f'{SUBJECT_ID} — Calibration & Risk–Coverage [{SPLIT_TYPE.upper()}]', y=0.98, fontsize=12)
plt.tight_layout(rect=[0,0,1,0.96])
save_path_1x4 = OUTPUT_DIR / f'{SUBJECT_ID}_{SPLIT_TYPE}_combined_1x4.png'
plt.savefig(save_path_1x4, dpi=300, bbox_inches='tight')
print(f"✓ 已保存合并图（1×4）: {save_path_1x4.name}")
plt.close(fig)


In [ ]:
# ====== C. Figure 4：Aggregated Soft CM + Long-tail + L1 Error ======
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import savgol_filter
from collections import defaultdict
from scipy.stats import spearmanr

print("Generating Figure 4: Advanced Analysis...")

# 依赖外部：N_CLASSES, fold_dirs, LABEL_NAMES, SUBJECT_ID, gt_proba, region_mask, OUTPUT_DIR
n_classes = int(N_CLASSES)

# ---------- 1) 读取并聚合软混淆 ----------
soft_cm_list = []
for fold_dir in fold_dirs:
    p = Path(fold_dir) / 'soft_confusion_test.csv'
    if p.exists():
        cm = np.loadtxt(p, delimiter=',')
        if cm.shape == (n_classes, n_classes):
            soft_cm_list.append(cm)
        else:
            print(f"  ⚠️ {fold_dir} 形状异常: {cm.shape}")

if len(soft_cm_list) == 0:
    aggregated_soft_cm = None
    print("❌ 没有发现 soft_confusion_test.csv，跳过 4a。")
else:
    aggregated_soft_cm = np.mean(soft_cm_list, axis=0)
    # 行归一（true class 条件分布）
    row_sum = aggregated_soft_cm.sum(axis=1, keepdims=True) + 1e-12
    aggregated_soft_cm = aggregated_soft_cm / row_sum
    print(f"✓ 聚合软混淆完成: {len(soft_cm_list)} 折")

# ---------- 2) 计算 per-class 指标与体素体量 ----------
stats = {"class_id": np.arange(n_classes), "class_name": [], "recall": [], "precision": [], "f1": [], "volume": []}

if aggregated_soft_cm is not None:
    # 按行归一后，对角线就是 recall
    recall = np.diag(aggregated_soft_cm)

    # 列归一得到 precision（pred 条件），为避免零除单独计算
    col_sum = aggregated_soft_cm.sum(axis=0, keepdims=True) + 1e-12
    prec_mat = aggregated_soft_cm / col_sum
    precision = np.diag(prec_mat)

    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    stats["recall"] = recall.tolist()
    stats["precision"] = precision.tolist()
    stats["f1"] = f1.tolist()
else:
    stats["recall"] = [np.nan]*n_classes
    stats["precision"] = [np.nan]*n_classes
    stats["f1"] = [np.nan]*n_classes

# 体素体量：用当前受试者的 soft label（真实数据，不做随机）
if 'gt_proba' in globals() and 'region_mask' in globals():
    roi_mask = (region_mask > 0)
    vol = np.einsum('ijkc,ijk->c', gt_proba, roi_mask.astype(float))
    stats["volume"] = vol.tolist()
else:
    print("❌ 缺少 gt_proba/region_mask，无法计算体量；该面板将提示缺失。")
    stats["volume"] = [np.nan]*n_classes

for cid in range(n_classes):
    stats["class_name"].append(LABEL_NAMES.get(cid, f'Class_{cid}'))

df = pd.DataFrame(stats)

# ---------- 3) 聚合 L1 误差（仅真实读取） ----------
class_mass_errors = defaultdict(list)
for fold_dir in fold_dirs:
    mp = Path(fold_dir) / 'metrics_test.json'
    if mp.exists():
        try:
            with open(mp, 'r') as f:
                metrics = json.load(f)
            for ei in metrics.get('class_mass_top_errors', []):
                class_mass_errors[ei['class_id']].append(float(ei['abs_error']))
        except Exception as e:
            print(f"  ⚠️ 读取 {mp} 失败: {e}")

avg_l1 = [np.mean(class_mass_errors[c]) if c in class_mass_errors and len(class_mass_errors[c])>0 else np.nan
          for c in range(n_classes)]

# ---------- 4) 选择展示次序（这里按体量排序，更诚实；若你有邻接矩阵可替换） ----------
valid_vol_mask = np.isfinite(df['volume'].values)
if valid_vol_mask.any():
    order_by_vol = np.argsort(df.loc[valid_vol_mask, 'volume'].values)[::-1]
    ordered_ids = df.index[valid_vol_mask][order_by_vol].values
    top_k = min(50, len(ordered_ids))
    anatomical_order = ordered_ids[:top_k]
else:
    anatomical_order = np.arange(min(50, n_classes))
    print("⚠️ 未能计算体量，临时使用 ID 顺序（非随机）。")

# ---------- 5) 绘图 ----------
fig = plt.figure(figsize=(15, 5))
gs = fig.add_gridspec(1, 3, wspace=0.35)

# (a) 软混淆矩阵（行归一）
ax1 = fig.add_subplot(gs[0,0])
if aggregated_soft_cm is not None:
    cm_sub = aggregated_soft_cm[np.ix_(anatomical_order, anatomical_order)]
    im = ax1.imshow(cm_sub, aspect='auto', vmin=0, vmax=1, cmap='hot')
    ax1.set_title(f'(a) Aggregated soft confusion (row‑normalized)\nTop {len(anatomical_order)} classes by volume', fontsize=10)
    ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
    cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    cbar.set_label('P(pred | true)')
    # 每 10 类打一个刻度
    ticks = np.arange(0, len(anatomical_order), 10)
    ax1.set_xticks(ticks); ax1.set_yticks(ticks)
else:
    ax1.text(0.5, 0.5, 'Soft confusion not available', ha='center', va='center')
    ax1.set_axis_off()

# (b) 长尾：体量 vs Top‑1 recall（Spearman ρ）
ax2 = fig.add_subplot(gs[0,1])
df_valid = df[np.isfinite(df['volume']) & np.isfinite(df['recall'])]
if len(df_valid) > 0:
    x = df_valid['volume'].values
    y = df_valid['recall'].values
    sc = ax2.scatter(x, y, s=18, edgecolor='k', linewidth=0.3)
    # 平滑趋势（Savitzky–Golay）
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    if len(xs) >= 11:
        win = (len(xs)//3) | 1  # odd
        win = max(11, min(win, 51) | 1)
        y_smooth = savgol_filter(ys, window_length=win, polyorder=2)
        ax2.plot(xs, y_smooth, lw=2, label='Smoothed trend (Savitzky–Golay)')
    rho, p = spearmanr(x, y)
    ax2.set_xscale('log')
    ax2.set_xlabel('Class volume (log scale)'); ax2.set_ylabel('Top‑1 recall')
    ax2.set_title(f'(b) Long‑tail effect  (Spearman ρ={rho:.3f}, p={p:.2g})', fontsize=10)
    ax2.legend(frameon=False, loc='lower right')
else:
    ax2.text(0.5, 0.5, 'Volume/recall not available', ha='center', va='center')
    ax2.set_axis_off()

# (c) 类质量 L1 误差分布（仅真实读取）
ax3 = fig.add_subplot(gs[0,2])
valid_err = np.array([e for e in avg_l1 if np.isfinite(e)])
if valid_err.size > 0:
    ax3.hist(valid_err, bins=30, edgecolor='k')
    ax3.axvline(valid_err.mean(), color='C1', ls='--', lw=1.5, label=f'Mean={valid_err.mean():.1f}')
    ax3.axvline(np.median(valid_err), color='C2', ls='--', lw=1.5, label=f'Median={np.median(valid_err):.1f}')
    ax3.set_xlabel('Per‑class L1 error of probability mass')
    ax3.set_ylabel('Number of classes')
    ax3.set_title('(c) Class‑mass L1 error distribution', fontsize=10)
    ax3.legend(frameon=False)
else:
    ax3.text(0.5, 0.5, 'L1 error not available', ha='center', va='center')
    ax3.set_axis_off()

plt.tight_layout()
save_path = OUTPUT_DIR / 'figure4_advanced_analysis.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {save_path}")
plt.close(fig)
